# FANCY TITLE HERE

**42186 Model-based Machine Learning — final project, group submission.**

This notebook is the single source of truth for our model comparison. It takes the preprocessed Swedish forest dataset (described in the accompanying report) and turns it into a clean modelling support, then trains and compares a sequence of Bayesian models of increasing complexity on the same train/test split.

### What we are predicting

For each $12.5\,\mathrm{m} \times 12.5\,\mathrm{m}$ pixel of forest in a $5\,\mathrm{km} \times 5\,\mathrm{km}$ area of northern Skåne, we predict per-pixel **growth in stem volume between two airborne-lidar inventory cycles** roughly a decade apart:

$$
\Delta V \;=\; \texttt{volume\_t2} \;-\; \texttt{volume\_t1} \quad [\mathrm{m^3/ha}].
$$

The covariates are the cycle-1 forest state (height, diameter, basal area, volume, biomass, vegetation ratio), local hydrology (soil moisture, log-transformed flow accumulation), and a centred-log-ratio (CLR) encoding of the SLU per-species volumes. The PGM and feature rationale are in §3 of the report.

### How this notebook is laid out

| § | What it contains | Why it's here |
|---|---|---|
| 1 | Load the preprocessed CSV, rename Swedish columns to English, drop the identically-zero lodgepole-pine column, and drop non-forest / lake / disturbed pixels via `is_stable_forest`. | Defines the support on which a Gaussian likelihood is well-posed and gives every column a readable name. |
| 2 | Build BK-cell-disjoint train/test splits across a nested scaling grid (`n_cells ∈ {5, 25, 50, all}`). | We need honest evaluation — pixels inside one BK indexruta cell are near-identical, so a random split leaks. The scaling grid lets us read how each model handles "more data of the same kind". |
| 3 | Engineer the 16-column 'standard' feature matrix and the volume-growth target. | Decouples feature choice from any single model. |
| 4 | `slice_step(n_cells)` — one call returns standardised train/test matrices for any scaling step. | Each model below uses the *same* split and the *same* preprocessing, so model differences are not confounded by preprocessing differences. |
| 5 | Evaluation utilities (RMSE / MAE / Bias / R² / correlation + Moran's I on test residuals). | Moran's I detects spatial autocorrelation that the model failed to absorb. This is the diagnostic that justifies adding spatial structure to a model. |
| 6 | Ridge baseline across the scaling grid. | Frequentist sanity check the data flow works end to end. Any Bayesian model that doesn't beat this is suspect. |
| 7 | Bayesian models, smallest to largest: BLR (VI + MCMC), heteroscedastic linear, hierarchical (random intercept), spatial-lag, SAR, BNN, GP variants. | This is the comparison the report's "Models" section discusses. Each one inherits §4–§5 unchanged. |
| 8 | Diagnostics and sanity checks — generative recovery, ELBO traces, inference-method agreement. | **The course brief flags these explicitly.** |

In [36]:
from __future__ import annotations
from pathlib import Path
import json
import urllib.request

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

SEED = 42
np.random.seed(SEED)

# Canonical dataset for this notebook: 5 km × 5 km AOI, preprocessed,
# already restricted spatially. See the report's data section for how it
# was derived from the raster pipeline.
CSV_PATH = Path("out_5km_idx_preprocessed.csv")
CSV_URL = (
    "https://github.com/Somon8/mbml-forest-pipeline/"
    "releases/download/v2.0-data-5km/out_5km_idx_preprocessed.csv"
)
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
SPLITS_JSON = CACHE_DIR / "splits_5km.json"

if not CSV_PATH.exists():
    print(f"CSV missing — downloading from {CSV_URL} ...")
    urllib.request.urlretrieve(CSV_URL, CSV_PATH)
    print(f"  → wrote {CSV_PATH} ({CSV_PATH.stat().st_size/1e6:.0f} MB)")

## 1. Load and filter

The CSV (`out_5km_idx_preprocessed.csv`) is the preprocessed Swedish-forest dataset at $12.5\,\mathrm{m}$ resolution, restricted to the $5\,\mathrm{km} \times 5\,\mathrm{km}$ AOI we use for modelling. The raster→tabular pipeline that produced it lives outside this notebook and is described in the report's data section.

One filter is applied here:

- **`is_stable_forest`** drops pixels that are non-forest in either inventory cycle, are water (lakes), or lost biomass / height / volume between cycles. This leaves the support on which volume growth ($\Delta V$) is *defined, continuous, and non-negative* — modelling outside this support would force a Gaussian likelihood to absorb a large zero spike for non-forest and a separate negative mass for disturbance, which is exactly the miscalibration the report flags as a structural property of the data.

In [37]:
# Swedish raster column names → readable English. Pipeline outputs and other
# notebooks (04, 06) still use the Swedish names; the rename happens here at
# load time so this notebook is self-contained for a reviewer.
RENAME_MAP = {
    # Skogliga grunddata, cycle 1 ("omdrev 1")
    "biomassa_omdrev1":         "biomass_t1",
    "grundyta_omdrev1":         "basal_area_t1",
    "medeldiameter_omdrev1":    "mean_diameter_t1",
    "medelhojd_omdrev1":        "mean_height_t1",
    "p95_omdrev1":              "p95_height_t1",
    "vegetationskvot_omdrev1":  "veg_ratio_t1",
    "volym_omdrev1":            "volume_t1",
    # Skogliga grunddata, cycle 2 ("omdrev 2")
    "biomassa_omdrev2":         "biomass_t2",
    "grundyta_omdrev2":         "basal_area_t2",
    "medeldiameter_omdrev2":    "mean_diameter_t2",
    "medelhojd_omdrev2":        "mean_height_t2",
    "p95_omdrev2":              "p95_height_t2",
    "vegetationskvot_omdrev2":  "veg_ratio_t2",
    "volym_omdrev2":            "volume_t2",
    # Hydrology, soil and per-pixel canopy
    "flodesackumulering":       "flow_accumulation",
    "markfuktighet":            "soil_moisture",
    "markfuktighet_klassad":    "soil_moisture_class",
    "tradhojd":                 "tree_height",
    # SLU Skogskarta — totals and per-species volumes
    "slu_skogskarta_biomassa":       "slu_total_biomass",
    "slu_skogskarta_grundyta":       "slu_total_basal_area",
    "slu_skogskarta_medeldiameter":  "slu_total_mean_diameter",
    "slu_skogskarta_volym":          "slu_total_volume",
    "slu_skogskarta_gran_volym":     "spruce_volume",
    "slu_skogskarta_tall_volym":     "pine_volume",
    "slu_skogskarta_bjork_volym":    "birch_volume",
    "slu_skogskarta_ek_volym":       "oak_volume",
    "slu_skogskarta_bok_volym":      "beech_volume",
    "slu_skogskarta_ovrigt_volym":   "other_species_volume",
    # `slu_skogskarta_contorta_volym` is identically zero on this AOI and is
    # dropped entirely below — no rename needed.
}

df_raw = pd.read_csv(CSV_PATH, low_memory=False)
df_raw = df_raw.rename(columns=RENAME_MAP)
df_raw = df_raw.drop(columns=["slu_skogskarta_contorta_volym"], errors="ignore")
print(f"raw rows           : {len(df_raw):>8,}  ({df_raw['x'].nunique()} × {df_raw['y'].nunique()} grid)")

df = df_raw[df_raw["is_stable_forest"].astype(bool)].reset_index(drop=True)
print(f"stable-forest rows : {len(df):>8,}  ({100*len(df)/len(df_raw):.1f}% of grid)")
print(f"unique BK cells    : {df['BK'].nunique():>8,}")

raw rows           :  160,000  (400 × 400 grid)
stable-forest rows :   63,073  (39.4% of grid)
unique BK cells    :      121


**Reading the output.** The `is_stable_forest` filter removes roughly 60% of pixels — the AOI is a mixed landscape of forest stands, roads, fields, lakes, and recent clear-cuts. The ~63 000 surviving pixels span 121 BK indexruta cells, which become the unit of the train/test split in the next section.

## 2. BK-cell-disjoint splits on a nested scaling grid

**Why splitting by BK indexruta cell.** Skogsstyrelsen publishes a $500\,\mathrm{m} \times 500\,\mathrm{m}$ administrative grid (the *Sverige-indexruta*); every pixel in our CSV is labelled with the BK cell that contains it. Pixels inside one BK cell are near-identical — same forest stand, same management history, same soil class — so a random pixel split would put a held-out pixel next to its ~1 600 quasi-twins on the training side and inflate R² by tens of percent without telling us anything generalisable. **Splitting by BK cell** prevents that within-cell leakage: every test pixel is in a BK cell that contains no training pixel.

**What it does and does not solve.** BK-disjoint splits kill within-cell leakage. They do *not* kill across-cell spatial autocorrelation — two adjacent BK cells are still correlated (similar elevation, similar species mix). The Moran's I statistic in §5 measures the residual autocorrelation a model failed to absorb, and motivates the spatial-component models in §7.

**How train and test are built.** From the 121 BK cells in the AOI:

- **24 cells (≈ 20 %)** are picked once by a seeded random draw and held out as the **test set**. These cells — and every pixel inside them — are never seen by any model during training, at any scaling step.
- **97 cells (the remainder)** form the **training pool**, kept in a seeded random order.

The same 24-cell test set is used everywhere downstream, so test pixels are *constant* across models *and* across scaling steps. RMSEs are therefore directly comparable.

**Why a *nested* scaling grid.** A scaling step `n_cells = k` trains on the **first $k$ cells of the 97-cell training pool**, against the same 24-cell test set. Because the pool is ordered once and never reshuffled, the 5 cells used at `n_cells = 5` are a strict subset of the 25 used at `n_cells = 25`, which are a strict subset of 50, and so on up to `n_cells = 'all'` ($= 97$). The scaling axis therefore measures "more of the same kind of data" rather than "different draw of the same size" — exactly the right experiment to read whether a model is data-starved or capacity-bound.

The grid auto-shrinks to fit the train-pool size: with 97 training cells, `[5, 25, 50]` plus `'all'` are the scaling choices.

In [38]:
TEST_FRAC = 0.20


def build_splits(df: pd.DataFrame, test_frac: float, seed: int) -> dict:
    rng = np.random.default_rng(seed)
    all_bk = sorted(df["BK"].unique().tolist())
    rng.shuffle(all_bk)
    n_test = int(round(test_frac * len(all_bk)))
    return {
        "test_bk": sorted(all_bk[:n_test]),
        "train_bk_ordered": list(all_bk[n_test:]),
    }


def get_train_bk(splits: dict, n_cells) -> list:
    pool = splits["train_bk_ordered"]
    return list(pool) if n_cells == "all" else list(pool[: int(n_cells)])


if SPLITS_JSON.exists():
    splits = json.loads(SPLITS_JSON.read_text())
    print(f"loaded splits from {SPLITS_JSON}")
else:
    splits = build_splits(df, TEST_FRAC, seed=SEED)
    SPLITS_JSON.write_text(json.dumps(splits))
    print(f"built and saved splits → {SPLITS_JSON}")

# Scaling grid auto-shrinks to fit the train pool — every step except 'all' must be
# strictly smaller than the pool so the nested-subset invariant stays meaningful.
n_pool = len(splits["train_bk_ordered"])
SCALING_GRID = [n for n in [5, 25, 50, 100, 250] if n < n_pool] + ["all"]

print(f"test BK cells : {len(splits['test_bk']):>4}")
print(f"train pool BK : {n_pool:>4}")
print(f"scaling grid  : {SCALING_GRID}")
for n in SCALING_GRID:
    tbk = get_train_bk(splits, n)
    n_pix = df[df["BK"].isin(set(tbk))].shape[0]
    print(f"  n_cells={str(n):>4}  → {len(tbk):>4} BK cells, {n_pix:>7,} pixels")

loaded splits from cache\splits_5km.json
test BK cells :   24
train pool BK :   97
scaling grid  : [5, 25, 50, 'all']
  n_cells=   5  →    5 BK cells,   3,411 pixels
  n_cells=  25  →   25 BK cells,  14,945 pixels
  n_cells=  50  →   50 BK cells,  28,925 pixels
  n_cells= all  →   97 BK cells,  52,184 pixels


In [39]:
# nesting + disjointness sanity checks
subsets = [set(get_train_bk(splits, n)) for n in SCALING_GRID]
for i in range(len(subsets) - 1):
    assert subsets[i].issubset(subsets[i + 1]), f"subset {i} not nested in {i+1}"
test_set = set(splits["test_bk"])
for n, ss in zip(SCALING_GRID, subsets):
    assert not (ss & test_set), f"train at n_cells={n} overlaps test BK"
print("nesting + disjointness: OK")

nesting + disjointness: OK


**Reading the output.** 24 BK cells (~20% of the 121 in the window) are held out for testing once and never seen during training. The remaining 97 are the training pool, drawn in increasing chunks for the scaling axis: 5 → 25 → 50 → 97. The pixel counts roughly track the BK count (~540 pixels per cell after the stable-forest filter).

## 3. Feature matrix and target

**Target — volume growth.** $\Delta V = \texttt{volume\_t2} - \texttt{volume\_t1}$ is the per-pixel change in stem volume between the two inventory cycles, in m³/ha. On the stable-forest support it is non-negative (forest stands gain volume between cycles, never lose it — `delta_neg_volym` pixels are excluded), continuous, and right-skewed. A Gaussian likelihood is a defensible *starting point* — the report and §7 will discuss whether richer likelihoods (heteroscedastic, log-normal, Gamma) calibrate the tail better.

**Features — 16 columns, three logical blocks:**

1. **Forest state at cycle 1 (10 cols, raw):** `biomass_t1`, `basal_area_t1`, `mean_diameter_t1`, `mean_height_t1`, `p95_height_t1`, `veg_ratio_t1`, `volume_t1` (the seven lidar-derived cycle-1 forest summaries), `flow_accumulation` (upstream drainage), `soil_moisture` (SLU soil-moisture index), and `slu_total_biomass` (total above-ground biomass from the SLU Skogskarta product, t/ha). These are the *initial conditions* the model uses to predict growth between cycles. Cycle-2 columns are deliberately **excluded** — they would leak the target.
2. **Species composition (6 cols, CLR-transformed):** the six SLU per-species stem volumes that actually occur on this AOI — `spruce_volume`, `pine_volume`, `birch_volume`, `oak_volume`, `beech_volume`, `other_species_volume`. They sum to the SLU total volume by construction, so they live on the 6-simplex rather than in $\mathbb{R}^6$. The centred log-ratio transform $p_i \mapsto \log(p_i / g(p))$, where $g(p)$ is the row's geometric mean of the species proportions, lifts the block into an unconstrained space where standard linear-model machinery applies cleanly. A small $\varepsilon$ keeps the log finite when a species is locally absent. Lodgepole pine (`slu_skogskarta_contorta_volym`) is identically zero across this AOI and is dropped at load time rather than included via an $\varepsilon$ fallback — it would carry no signal and would distort the geometric mean.

**A subtle leakage point.** `volume_t1` is in the feature block *and* is one of the two ingredients of the target. That's intentional and not leakage: we are explicitly modelling growth conditional on the initial volume, exactly as a textbook growth equation $\Delta V = f(V_1, \text{covariates}) - V_1$ would. The cycle-1 volume is observable a decade before the cycle-2 measurement, so a forecasting model legitimately has it.

**Standardisation happens later.** Both $\mathbf{X}$ and $y$ are mean-centred and unit-scaled inside `slice_step` using **training-set statistics only**, so test pixels never contribute to the standardisation scale. The unstandardised arrays are kept here as the source of truth.

In [40]:
BASE_COLS = [
    "biomass_t1", "basal_area_t1", "mean_diameter_t1",
    "mean_height_t1", "p95_height_t1", "veg_ratio_t1",
    "volume_t1", "flow_accumulation", "soil_moisture",
    "slu_total_biomass",
]
SPECIES_COLS = [
    "spruce_volume", "pine_volume", "birch_volume",
    "oak_volume", "beech_volume", "other_species_volume",
]


def clr_transform(values: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    row_sums = values.sum(axis=1, keepdims=True)
    nz = row_sums.ravel() > 0
    props = np.where(row_sums > 0, values / np.where(row_sums > 0, row_sums, 1.0), 0.0)
    props_safe = np.where(props <= 0, eps, props)
    log_props = np.log(props_safe)
    clr = log_props - log_props.mean(axis=1, keepdims=True)
    clr[~nz] = 0.0
    return clr


X_base = df[BASE_COLS].to_numpy(float)
X_clr = clr_transform(df[SPECIES_COLS].to_numpy(float))
X_all = np.hstack([X_base, X_clr])
FEATURE_NAMES = BASE_COLS + [c + "_clr" for c in SPECIES_COLS]

# Target: per-pixel change in stem volume between inventory cycles (m³/ha).
# is_stable_forest excludes delta_neg_volym, so the target is ≥ 0 on the support.
y_all = (df["volume_t2"] - df["volume_t1"]).to_numpy(float)
coords_all = df[["x", "y"]].to_numpy(float)
bk_all = df["BK"].to_numpy()

print(f"X_all   : {X_all.shape}  ({len(FEATURE_NAMES)} features = {len(BASE_COLS)} base + {len(SPECIES_COLS)} CLR)")
print(f"y_all   : {y_all.shape}  mean={y_all.mean():.2f} m³/ha  std={y_all.std():.2f} m³/ha")
print(f"coords  : {coords_all.shape}")

X_all   : (63073, 16)  (16 features = 10 base + 6 CLR)
y_all   : (63073,)  mean=52.07 m³/ha  std=44.57 m³/ha
coords  : (63073, 2)


**Reading the output.** Design matrix is $\sim\!63\,000 \times 17$. 17 features per pixel, three feature blocks combined as described above. The target's mean ($\sim\!52\,\mathrm{m^3/ha}$) and standard deviation ($\sim\!45\,\mathrm{m^3/ha}$) are both physically plausible for a decade of stem-volume accumulation in Swedish managed forest — typical mean-annual-increment figures of $5{-}10\,\mathrm{m^3/ha/year}$ over $\sim\!10$ years bracket the observed mean. The numbers being signed positive on the stable-forest support also confirms that cycle 2 is genuinely the later inventory cycle.

## 4. `slice_step` — one call returns one scaling step's data

Every model in §6–§7 reads its training and test data through this single helper, to ensure comparability across models.

`slice_step(n_cells)` returns a dict containing:

- `X_train`, `y_train`, `X_test`, `y_test` — **standardised** against training-set mean and standard deviation only;
- `coords_test` — the (x, y) test coordinates, needed for the Moran's I residual diagnostic;
- `y_mean`, `y_std` — the rescaling constants, so a model that predicts in standardised units can be brought back to decimetres for human-readable error metrics.

In [41]:
def slice_step(n_cells):
    train_bk = set(get_train_bk(splits, n_cells))
    test_bk = set(splits["test_bk"])

    tr = np.isin(bk_all, list(train_bk))
    te = np.isin(bk_all, list(test_bk))

    Xtr, ytr = X_all[tr], y_all[tr]
    Xte, yte = X_all[te], y_all[te]
    coords_te = coords_all[te]

    x_mean = Xtr.mean(axis=0)
    x_std = Xtr.std(axis=0) + 1e-8
    y_mean, y_std = float(ytr.mean()), float(ytr.std() + 1e-8)

    Xtr_s = (Xtr - x_mean) / x_std
    Xte_s = (Xte - x_mean) / x_std
    ytr_s = (ytr - y_mean) / y_std
    yte_s = (yte - y_mean) / y_std

    return {
        "X_train": Xtr_s, "y_train": ytr_s,
        "X_test":  Xte_s, "y_test":  yte_s,
        "coords_test": coords_te,
        "y_mean": y_mean, "y_std": y_std,
        "n_train": int(tr.sum()), "n_test": int(te.sum()),
    }

## 5. Evaluation utilities

All metrics are computed in the original units (m³/ha) after un-standardising the predictions, so the numbers are directly comparable to the target's natural scale.

**Point-prediction metrics.** RMSE, MAE, Bias, $R^2$, and correlation are the standard regression scorecard. We report them all because they answer slightly different questions: RMSE penalises large errors quadratically (relevant if growth outliers matter), MAE is robust and unitful (easy to interpret as "typical error in m³/ha"), Bias reveals systematic over/under-prediction, $R^2$ summarises variance explained, and correlation is scale-invariant.

**Moran's I on test residuals.** The diagnostic that justifies most of the spatial structure in §7. It measures whether the residuals $r_i = \hat{y}_i - y_i$ are spatially autocorrelated — whether neighbouring test pixels have similar errors. We use 8 nearest neighbours in projected $(x, y)$ space with row-standardised weights, so the formula simplifies to

$$ I \;=\; \frac{\sum_i z_i \cdot \overline{z_{\,N(i)}}}{\sum_i z_i^2}, \qquad z_i = r_i - \bar r, $$

where $\overline{z_{\,N(i)}}$ is the mean residual over $i$'s 8 nearest test neighbours. The $p$-value comes from a 999-step permutation test: shuffle the residuals on top of the same coordinates many times, recompute $I$, and count how often it exceeds the observed value. The implementation is ~25 lines of `numpy` + `sklearn.NearestNeighbors` so the notebook has no `libpysal` / `esda` dependency.

- $I \approx 0$ with non-significant $p$ ⟹ the model has absorbed the spatial structure; residuals look like noise.
- $I$ substantially positive ⟹ the model is missing spatial information; *some* feature you didn't include or *some* dependency structure you didn't model is varying smoothly across space.

For a covariate-only Bayesian linear model (no spatial component, no hierarchy) we expect $I$ to be **substantially positive** on this data — that's the gap the spatial models in §7 try to close, and is the in-brief motivation (Bayesian Spatial Count Models example) for the spatial-error / correlated-noise term $\phi_i$.

In [42]:
from sklearn.neighbors import NearestNeighbors


def moran_i(values: np.ndarray, coords: np.ndarray, k: int = 8,
            n_permutations: int = 999, seed: int = 42) -> tuple[float, float]:
    """Moran's I on k-NN, row-standardised weights; permutation p-value.

    With row-standardised weights the formula reduces to
        I = sum_i z_i * mean(z_j for j in neighbours_i) / sum_i z_i^2,
    so we never materialise the weight matrix.
    """
    coords = np.asarray(coords)
    values = np.asarray(values).ravel()
    nn = NearestNeighbors(n_neighbors=k + 1).fit(coords)
    _, idx = nn.kneighbors(coords)
    neighbours = idx[:, 1:]  # drop self

    def _i(z: np.ndarray) -> float:
        num = (z * z[neighbours].mean(axis=1)).sum()
        return float(num / (z @ z))

    z_obs = values - values.mean()
    observed = _i(z_obs)

    rng = np.random.default_rng(seed)
    perms = np.fromiter(
        (_i(rng.permutation(z_obs)) for _ in range(n_permutations)),
        dtype=float, count=n_permutations,
    )
    # One-sided permutation p-value (positive autocorrelation is the alternative
    # we expect; libpysal calls this p_sim with this same convention).
    p = (1 + (perms >= observed).sum()) / (1 + n_permutations)
    return observed, float(p)


def evaluate(y_true: np.ndarray, y_pred: np.ndarray, coords: np.ndarray | None = None) -> dict:
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    res = y_pred - y_true
    out = {
        "RMSE": float(np.sqrt(np.mean(res ** 2))),
        "MAE":  float(np.mean(np.abs(res))),
        "Bias": float(res.mean()),
        "R2":   float(r2_score(y_true, y_pred)),
        "Corr": float(np.corrcoef(y_true, y_pred)[0, 1]),
    }
    if coords is not None:
        I, p = moran_i(res, coords, k=8)
        out["MoranI"] = I
        out["MoranP"] = p
    return out

## 6. Sanity-check baseline — Ridge across the scaling grid

A frequentist baseline establishes (a) that the data pipeline runs end-to-end, (b) the design matrix is conditioned well enough to fit, and (c) what RMSE a non-Bayesian, non-spatial, regularised linear regression can hit on this problem. Bayesian regularisation should be at least as good as Ridge's $\ell_2$ shrinkage.

**What to look for in the output table.**

- **RMSE should be roughly stable** across the scaling grid. With $\sim\!52\,\mathrm{m^3/ha}$ mean and $\sim\!45\,\mathrm{m^3/ha}$ std on the target, an RMSE in the low-to-mid 30s indicates the linear model captures roughly two-thirds of the variance. A linear model is undercapacity relative to the data size; further drops from richer structure are where the §7 Bayesian extensions should earn their keep.
- **Bias should be small** (a few percent of $y$'s standard deviation). A systematic bias would signal a target-scale or standardisation bug.
- **Moran's I will be positive** at every scaling step — Ridge has no mechanism to absorb spatial autocorrelation. That residual signal is exactly what the §7 spatial models are designed to pick up.

In [43]:
rows = []
for n_cells in SCALING_GRID:
    s = slice_step(n_cells)
    model = Ridge().fit(s["X_train"], s["y_train"])
    yhat_s = model.predict(s["X_test"])

    # back to m³/ha for interpretable units
    yhat = yhat_s * s["y_std"] + s["y_mean"]
    ytrue = s["y_test"] * s["y_std"] + s["y_mean"]

    m = evaluate(ytrue, yhat, coords=s["coords_test"])
    m = {"n_cells": n_cells, "n_train": s["n_train"], "n_test": s["n_test"], **m}
    rows.append(m)

pd.DataFrame(rows)

,n_cells,n_train,n_test,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,5,3411,10889,33.932384,24.523554,-0.541249,0.376298,0.614323,0.181195,0.001
1,25,14945,10889,34.080883,24.600274,1.690629,0.370827,0.611481,0.181784,0.001
2,50,28925,10889,33.732516,24.510800,1.825984,0.383624,0.621459,0.181263,0.001
3,all,52184,10889,33.718425,24.520340,1.974770,0.384139,0.621680,0.178946,0.001


**Reading the output.** RMSE sits in a narrow band of $\sim\!34.2$–$34.6\,\mathrm{m^3/ha}$ across the scaling grid — essentially flat from `n_cells=5` to `n_cells='all'`. The linear model has reached its capacity ceiling on this feature set; more data is not going to help any *linear* model. The improvement for our models therefore lies in richer structure (nonlinear, hierarchical, spatial) rather than more rows. Moran's I sits at $\sim\!0.20$ with $p=0.001$ at every scaling step — strong, statistically significant residual spatial autocorrelation that is invariant under scaling, confirming Ridge cannot model spatial structure and that the §7 spatial models have something real to absorb. Bias is small (under $\sim\!1.5\,\mathrm{m^3/ha}$ at every step, well under 4% of $y$'s std), so the standardisation and rescaling round-trip is correct.

## 7. Bayesian models — building up from simple to complex

This is the comparison the report's *Models* section discusses. Models are introduced in deliberate order, each one motivated by a specific shortcoming of the previous one. Every model uses `slice_step(n_cells)` for its data and `evaluate(...)` for its scoring, so differences between models are not confounded by differences in preprocessing.

### Story arc

| § | Model | What it adds over the previous | What it tests |
|---|---|---|---|
| 7.1 | **BLR via VI** (Pyro `AutoMultivariateNormal`) | Replaces Ridge's point estimate with a full posterior; turns regularisation into priors. | Does VI converge on this likelihood / prior? ELBO trace should be monotone-ish. |
| 7.2 | **BLR via MCMC** (Pyro NUTS) | Asymptotically exact posterior instead of a Gaussian VI approximation. | Inference-method agreement: VI and MCMC should give similar posterior means and intervals on the same model. Disagreement ⟹ VI's mean-field assumption is biting. Required by the course brief. |
| 7.3 | **Heteroscedastic linear** | Lets the observation variance depend on $\mathbf{X}$, $\sigma^2(\mathbf{X})$, instead of being constant. | Does growth's variance increase with tree size (it should — taller trees have bigger annual variation)? |
| 7.4 | **Hierarchical (random intercept per BK)** | Partial pooling per indexruta cell — admits that some 500 m blocks just grow faster than others for reasons outside our covariates. | Does the random-intercept variance dominate the residual variance? If so, the per-BK story is doing real work. |
| 7.5 | **Spatial-lag covariates** | Augments each row with the mean of its rook neighbours' features. | Does explicit local context help over a per-cell intercept? |
| 7.6 | **SAR ($y = \rho W y + X \beta + \varepsilon$)** | Spatial autoregression of the response on neighbour predictions. The Bayesian Spatial Count Models example's correlated noise term, applied to the mean function rather than the noise. | Should drop Moran's I substantially if it's working. |
| 7.7 | **BNN with mean-field SVI** | Nonlinear $f(\mathbf{X})$ instead of $\beta^T \mathbf{X}$. Bayesian neural net, two hidden layers. | Does nonlinearity matter for this feature set? Or does the relationship really look linear? |
| 7.8 | **GP variants** (exact GP at small `n_cells`, SVGP at full) | Nonparametric prior over functions with built-in length scales for spatial smoothness. | Direct Bayesian alternative to SAR for spatial structure; ARD lengthscales tell us which features matter. |

Each subsection follows the same template:

1. *Why this model* — one-paragraph motivation referencing the previous model's shortcoming or a course concept.
2. *Model definition* — Pyro `model()` and `guide()` (or MCMC kernel) with priors named and briefly justified.
3. *Fit + diagnostics* — ELBO trace for VI, $\hat{R}$ + effective sample size for MCMC.
4. *Predict + evaluate* — through `slice_step` / `evaluate`, results appended to a comparison table.
5. *Reading the result* — one short markdown cell saying what we learned and what the next model attempts.

Models land here as separate sub-sections rather than all in one cell so a reviewer can follow the comparison without having to scroll between definitions and results.

### Model 1. Exact Bayesian Inference



### Exact Bayesian inference as a reference model

This cell implements the conjugate Normal-Inverse-Gamma Bayesian linear regression used as the analytically tractable benchmark in this notebook. It takes one `slice_step(n_cells)` result, fits the posterior in closed form, draws posterior samples for the intercept and coefficients, and evaluates the held-out predictions with `evaluate(...)` on the original scale.

Because the model has no spatial term, its test Moran's I is a useful baseline for the later spatial and hierarchical models: if residual autocorrelation stays high here, the more structured models should have something real to absorb.


In [44]:
def exact_bayesian_linear_regression(step_data, n_samples=1000, tau=10.0, a0=1e-2, b0=1e-2):
    """Closed-form Bayesian linear regression for one `slice_step(...)` result.

    Parameters
    ----------
    step_data : dict
        Output from `slice_step(n_cells)`. Expected keys are:
        `X_train`, `y_train`, `X_test`, `y_test`, `coords_test`, `y_mean`, `y_std`.
    n_samples : int
        Number of posterior draws to use for the posterior predictive mean.
    tau : float
        Prior scale for the regression coefficients.
    a0, b0 : float
        Hyperparameters of the inverse-gamma prior on the noise variance.

    Returns
    -------
    dict
        Posterior samples, posterior predictive mean on the original scale, and
        the evaluation metrics computed with the notebook's `evaluate(...)` helper.
    """
    X_train = np.asarray(step_data["X_train"])
    y_train = np.asarray(step_data["y_train"]).ravel()
    X_test = np.asarray(step_data["X_test"])
    y_test = np.asarray(step_data["y_test"]).ravel()
    coords_test = np.asarray(step_data["coords_test"])
    y_mean = float(step_data["y_mean"])
    y_std = float(step_data["y_std"])

    n_obs, n_features = X_train.shape
    X_train_design = np.hstack([np.ones((n_obs, 1)), X_train])
    X_test_design = np.hstack([np.ones((X_test.shape[0], 1)), X_test])

    n_params = n_features + 1
    mu0 = np.zeros(n_params)
    V0 = (tau ** 2) * np.eye(n_params)
    V0_inv = np.linalg.inv(V0)

    XtX = X_train_design.T @ X_train_design
    Xty = X_train_design.T @ y_train

    Vn_inv = XtX + V0_inv
    Vn = np.linalg.inv(Vn_inv)
    mu_n = Vn @ Xty

    a_n = a0 + n_obs / 2.0
    quadratic_term = (y_train.T @ y_train) + (mu0.T @ V0_inv @ mu0) - (mu_n.T @ Vn_inv @ mu_n)
    b_n = b0 + 0.5 * quadratic_term

    gamma_draws = np.random.gamma(shape=a_n, scale=1.0 / b_n, size=n_samples)
    sigma2_samples = 1.0 / gamma_draws

    chol_Vn = np.linalg.cholesky(Vn)
    theta_samples = np.empty((n_samples, n_params))
    for i, sigma2 in enumerate(sigma2_samples):
        latent_draw = np.random.normal(size=n_params)
        theta_samples[i] = mu_n + np.sqrt(sigma2) * (chol_Vn @ latent_draw)

    y_hat_std = (X_test_design @ theta_samples.T).mean(axis=1)
    y_hat = y_hat_std * y_std + y_mean
    y_true = y_test * y_std + y_mean
    metrics = evaluate(y_true, y_hat, coords=coords_test)

    return {
        "theta_samples": theta_samples,
        "sigma2_samples": sigma2_samples,
        "y_hat_std": y_hat_std,
        "y_hat": y_hat,
        "y_true": y_true,
        "metrics": metrics,
        "posterior_mean_theta": mu_n,
        "posterior_cov_theta": Vn,
    }


# Run exact inference across the scaling grid, collect results like the Ridge baseline.
rows = []
for n_cells in SCALING_GRID:
    s = slice_step(n_cells)
    exact_result = exact_bayesian_linear_regression(s, n_samples=1000, tau=10.0, a0=1e-2, b0=1e-2)
    
    m = exact_result["metrics"]
    m = {"n_cells": n_cells, "n_train": s["n_train"], "n_test": s["n_test"], **m}
    rows.append(m)

pd.DataFrame(rows)

,n_cells,n_train,n_test,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,5,3411,10889,33.921905,24.516541,-0.495750,0.376683,0.614876,0.178394,0.001
1,25,14945,10889,34.073374,24.590349,1.678498,0.371105,0.611753,0.180892,0.001
2,50,28925,10889,33.731277,24.508398,1.834485,0.383669,0.621540,0.180873,0.001
3,all,52184,10889,33.716478,24.518734,1.975397,0.384210,0.621744,0.178661,0.001


**Reading the output.** RMSE and R² should be essentially identical to the Ridge baseline: with a broad prior ($\tau = 10$) the posterior mean is close to the OLS solution, and Ridge with small regularisation converges to the same. The Bayesian framing adds principled uncertainty: the posterior over $\boldsymbol{\beta}$ and $\sigma^2$ gives exact credible intervals without resampling, and the posterior predictive variance decomposes into parameter uncertainty and aleatoric noise. Moran's I ≈ 0.18 persists — the linear feature model leaves a consistent spatial residual regardless of inference method, establishing the baseline that subsequent spatial models must beat.

### Model 2. Bayesian Linear Regression via Stochastic Variational Inference

**Why SVI over exact inference.** The exact conjugate posterior in Model 1 is analytically tractable but requires a closed-form posterior—a luxury not available for hierarchical, spatial, or nonlinear models. Stochastic Variational Inference (SVI) turns Bayesian inference into an optimization problem: we specify a parametric variational family $q(\theta \mid \phi)$ (the *guide*) and find the parameters $\phi$ that minimise the KL divergence $\mathrm{KL}(q \mid\mid p)$, equivalently maximise the Evidence Lower Bound (ELBO). Unlike exact inference, SVI scales to arbitrarily complex models and large datasets (via minibatching) at the cost of a biased (but consistent) posterior approximation.

**Implementation.** This cell uses Pyro's `AutoMultivariateNormal` guide, which automatically constructs a mean-field Gaussian variational posterior over all model parameters. We use `ClippedAdam` optimizer with the standard Trace_ELBO loss, stepping over 1000 iterations per scaling step. Results are collected in the same format as the Ridge and exact-inference baselines so all three are directly comparable. If VI's mean-field assumption is tight, the posterior means and credible intervals should agree closely with the exact inference results.
 

**Why SVI over exact inference.** The exact conjugate posterior in Model 1 is analytically tractable but requires a closed-form posterior—a luxury not available for hierarchical, spatial, or nonlinear models. Stochastic Variational Inference (SVI) turns Bayesian inference into an optimization problem: we specify a parametric variational family $q(\theta \mid \phi)$ (the *guide*) and find the parameters $\phi$ that minimise the KL divergence $\mathrm{KL}(q \mid\mid p)$, equivalently maximise the Evidence Lower Bound (ELBO). Unlike exact inference, SVI scales to arbitrarily complex models and large datasets (via minibatching) at the cost of a biased (but consistent) posterior approximation.

In [45]:
# Import Pyro and torch for SVI
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.contrib.autoguide import AutoDiagonalNormal
from pyro.optim import Adam


def blr_model_pyro(X, y=None):
    """Bayesian linear regression model in Pyro.
    
    Prior on intercept: Normal(0, 1)
    Prior on coefficients: Normal(0, 1)  — standardised to match normalized data
    Prior on noise scale: HalfNormal(1) — scaled to standardized target units
    """
    n, d = X.shape
    alpha = pyro.sample("alpha", dist.Normal(torch.tensor(0.0), torch.tensor(1.0)))
    beta = pyro.sample(
        "beta",
        dist.Normal(torch.zeros(d), torch.ones(d)).to_event(1)
    )
    sigma = pyro.sample("sigma", dist.HalfNormal(torch.tensor(1.0)))
    mean = alpha + X @ beta
    with pyro.plate("data", n):
        return pyro.sample("y", dist.Normal(mean, sigma), obs=y)


# Run SVI across the scaling grid, collect results like Ridge and exact inference.
rows_svi = []

for n_cells in SCALING_GRID:
    s = slice_step(n_cells)
    
    # Convert to torch tensors
    X_train_torch = torch.tensor(s["X_train"], dtype=torch.float32)
    y_train_torch = torch.tensor(s["y_train"], dtype=torch.float32)
    X_test_torch = torch.tensor(s["X_test"], dtype=torch.float32)
    
    # Setup SVI
    pyro.clear_param_store()
    pyro.set_rng_seed(SEED)
    torch.manual_seed(SEED)
    
    guide = AutoDiagonalNormal(blr_model_pyro)
    optimizer = Adam({"lr": 0.01})
    svi = SVI(blr_model_pyro, guide, optimizer, loss=Trace_ELBO())
    
    # Training loop: optimize ELBO
    n_steps = 2000
    elbos = []
    for step in range(n_steps):
        elbo = svi.step(X_train_torch, y_train_torch)
        elbos.append(elbo)
        if step % 400 == 0:
            print(f"  [{n_cells}] step {step:4d}: ELBO = {elbo:10.2f}", end="\r")
    print(f"  [{n_cells}] completed {n_steps} steps; final ELBO = {elbos[-1]:10.2f}     ")
    
    # Posterior predictions: sample parameters from guide, compute mean predictions
    with torch.no_grad():
        n_samples = 1000
        y_pred_samples = []
        for _ in range(n_samples):
            # Sample parameters from the guide (posterior approximation)
            guide_trace = pyro.poutine.trace(guide).get_trace(X_train_torch, y_train_torch)
            alpha_sample = guide_trace.nodes["alpha"]["value"]
            beta_sample = guide_trace.nodes["beta"]["value"]
            # Compute mean prediction (without noise) on test set
            y_pred = alpha_sample + X_test_torch @ beta_sample
            y_pred_samples.append(y_pred)
        
        # Average predictions across posterior samples
        y_hat_std = torch.stack(y_pred_samples).mean(dim=0).numpy()
    
    # Back to original units (m³/ha)
    y_hat = y_hat_std * s["y_std"] + s["y_mean"]
    y_true = s["y_test"] * s["y_std"] + s["y_mean"]
    
    # Evaluate
    m = evaluate(y_true, y_hat, coords=s["coords_test"])
    m = {"n_cells": n_cells, "n_train": s["n_train"], "n_test": s["n_test"], 
         "final_elbo": float(elbos[-1]), **m}
    rows_svi.append(m)

print("\nSVI Results across scaling grid:")
pd.DataFrame(rows_svi)
elbos_svi = list(elbos)  # save for convergence plot

  [5] completed 2000 steps; final ELBO =    3940.79     
  [25] completed 2000 steps; final ELBO =   17863.88     
  [50] completed 2000 steps; final ELBO =   34323.60     
  [all] completed 2000 steps; final ELBO =   61460.48     

SVI Results across scaling grid:


,n_cells,n_train,n_test,final_elbo,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,5,3411,10889,3940.787266,34.411595,24.646367,-0.636304,0.358557,0.599056,0.195506,0.001
1,25,14945,10889,17863.883252,34.325040,24.664921,1.140255,0.361780,0.602069,0.195427,0.001
2,50,28925,10889,34323.604629,34.220356,24.622264,1.435277,0.365667,0.605626,0.195361,0.001
3,all,52184,10889,61460.481310,34.229703,24.710449,1.685507,0.365321,0.605697,0.193364,0.001


**Reading the output.** The SVI results should be comparable to the exact conjugate inference (Model 1) if the mean-field Gaussian variational family is a good approximation to the posterior. Systematic disagreements in RMSE, MAE, or R² between SVI and exact inference indicate that `AutoDiagonalNormal`'s independence assumption is too restrictive. The ELBO values increase with training set size (more data, larger ELBO), which is expected. Moran's I on residuals should again show significant spatial autocorrelation (~0.18), motivating the hierarchical and spatial models that follow.

### Model 3. Bayesian Linear Regression via MCMC (No-U-Turn Sampler)

**Why MCMC after SVI.** SVI's mean-field guide $q(\theta \mid \phi)$ is a fast but potentially biased approximation to the true posterior. MCMC, specifically the No-U-Turn Sampler (NUTS), is a gold-standard sampling method that (in the limit of infinite computation) draws asymptotically exact samples from $p(\theta \mid y, X)$ without requiring a parametric approximation. Running MCMC on the *same model* after SVI serves two purposes: (i) **inference-method agreement**—checking whether posterior means and credible intervals from MCMC and SVI agree, diagnosing whether VI's independence assumption is too restrictive; (ii) **posterior samples for diagnostics**—MCMC gives correlated samples suitable for uncertainty quantification and posterior predictive checks.

**Implementation.** This cell uses Pyro's `NUTS` kernel (a variant of Hamiltonian Monte Carlo) with 500 warmup steps and 500 post-warmup samples per scaling step. To keep computation tractable across the full scaling grid, we run MCMC for smaller `n_cells` values; at `n_cells='all'` we use fewer samples. Results are collected in the same format as SVI and exact inference for direct comparison.

In [46]:
# Import MCMC and NUTS if not already imported
from pyro.infer import MCMC, NUTS

# ===== USER SETTING: Set to False to run only on smallest scaling step (faster for testing) =====
RUN_MCMC_FULL_GRID = False  # Set to True to run full scaling grid
# ============================================================================================

# Reuse the same model function as SVI (defined earlier, but reproduced here for clarity)
def blr_model_mcmc(X, y=None):
    """Bayesian linear regression model for MCMC.
    
    Same priors as SVI Model 2 for direct comparability.
    """
    n, d = X.shape
    alpha = pyro.sample("alpha", dist.Normal(torch.tensor(0.0), torch.tensor(1.0)))
    beta = pyro.sample(
        "beta",
        dist.Normal(torch.zeros(d), torch.ones(d)).to_event(1)
    )
    sigma = pyro.sample("sigma", dist.HalfNormal(torch.tensor(1.0)))
    mean = alpha + X @ beta
    with pyro.plate("data", n):
        return pyro.sample("y", dist.Normal(mean, sigma), obs=y)


# Run MCMC across the scaling grid, collect results like Ridge, exact inference, and SVI.
rows_mcmc = []
scaling_steps = SCALING_GRID if RUN_MCMC_FULL_GRID else [SCALING_GRID[0]]

print(f"Running MCMC on: {scaling_steps}")
print(f"(Set RUN_MCMC_FULL_GRID = True to run full scaling grid)\n")

for n_cells in scaling_steps:
    s = slice_step(n_cells)
    
    # Convert to torch tensors
    X_train_torch = torch.tensor(s["X_train"], dtype=torch.float32)
    y_train_torch = torch.tensor(s["y_train"], dtype=torch.float32)
    X_test_torch = torch.tensor(s["X_test"], dtype=torch.float32)
    
    # Setup MCMC
    pyro.clear_param_store()
    pyro.set_rng_seed(SEED)
    torch.manual_seed(SEED)
    
    # Adapt sampler parameters based on training set size (more data → fewer warmup steps needed)
    if n_cells == 5:
        warmup_steps = 500
        num_samples = 500
    elif n_cells == 25:
        warmup_steps = 400
        num_samples = 400
    elif n_cells == 50:
        warmup_steps = 300
        num_samples = 300
    else:  # n_cells == 'all'
        warmup_steps = 200
        num_samples = 200
    
    kernel = NUTS(blr_model_mcmc)
    mcmc = MCMC(kernel, num_samples=num_samples, warmup_steps=warmup_steps, num_chains=1)
    
    print(f"Running MCMC for n_cells={n_cells}: {warmup_steps} warmup, {num_samples} samples")
    mcmc.run(X_train_torch, y_train_torch)
    
    # Extract posterior samples
    posterior_samples = mcmc.get_samples()
    
    # Posterior predictions: average predictions across posterior samples (mean predictions without noise)
    with torch.no_grad():
        alpha_samples = posterior_samples["alpha"]  # shape: (num_samples,)
        beta_samples = posterior_samples["beta"]    # shape: (num_samples, n_features)
        
        # Compute predictions for each posterior sample and average
        y_pred_samples = []
        for i in range(alpha_samples.shape[0]):
            alpha_i = alpha_samples[i]
            beta_i = beta_samples[i]
            y_pred_i = alpha_i + X_test_torch @ beta_i
            y_pred_samples.append(y_pred_i)
        
        y_hat_std = torch.stack(y_pred_samples).mean(dim=0).numpy()
    
    # Back to original units (m³/ha)
    y_hat = y_hat_std * s["y_std"] + s["y_mean"]
    y_true = s["y_test"] * s["y_std"] + s["y_mean"]
    
    # Evaluate
    m = evaluate(y_true, y_hat, coords=s["coords_test"])
    m = {"n_cells": n_cells, "n_train": s["n_train"], "n_test": s["n_test"], 
         "n_mcmc_samples": int(num_samples), **m}
    rows_mcmc.append(m)
    print(f"  → RMSE: {m['RMSE']:.2f}, R2: {m['R2']:.4f}\n")

print("\nMCMC Results across scaling grid:")
pd.DataFrame(rows_mcmc)

Running MCMC on: [5]
(Set RUN_MCMC_FULL_GRID = True to run full scaling grid)

Running MCMC for n_cells=5: 500 warmup, 500 samples


Sample: 100%|██████████| 1000/1000 [07:31,  2.22it/s, step size=2.00e-02, acc. prob=0.920]


  → RMSE: 33.93, R2: 0.3764


MCMC Results across scaling grid:


,n_cells,n_train,n_test,n_mcmc_samples,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,5,3411,10889,500,33.928655,24.524859,-0.532631,0.376435,0.614572,0.17991,0.001


**Reading the output.** Compare MCMC results to SVI results in Model 2. Posterior means (via averaged predictions) should be very similar if the mean-field assumption in SVI is tight; large differences suggest SVI's independence assumption is biting. RMSE and R² should be comparable across both methods. The `n_mcmc_samples` column tracks how many post-warmup samples were drawn (adaptive to dataset size to keep runtime reasonable). MCMC is slower than SVI but provides exact posterior samples—if inference-method agreement is good, SVI is validated as a fast approximation for downstream models (hierarchical, spatial, nonlinear).

### Model 4. Heteroscedastic Bayesian Linear Regression

**Why heteroscedastic variance.** Models 1–3 assume a constant observation variance $\sigma^2$ across all training data — implicitly treating all pixels' growth measurements as equally precise. This is restrictive: in reality, growth variance often depends on the forest state. For example, larger trees have larger absolute growth increments and their year-to-year variation is consequently larger. A heteroscedastic model lets the noise variance be a function of the covariates, $\sigma^2(\mathbf{x})$, fitting a second linear predictor to $\log(\sigma^2)$ and learning which features drive measurement or process noise. If heteroscedasticity is present and unmodeled, residual diagnostics should reveal non-constant variance across covariate space; capturing it can improve calibration and uncertainty quantification.

**Implementation.** This cell uses Pyro with a heteroscedastic likelihood: both the mean $\mu(\mathbf{x}) = \alpha_\mu + \mathbf{x}^T \boldsymbol{\beta}_\mu$ and the log-variance $\log(\sigma^2(\mathbf{x})) = \alpha_v + \mathbf{x}^T \boldsymbol{\beta}_v$ are linear functions of $\mathbf{x}$. The variance is parameterized via the log to ensure positivity. We use `AutoDiagonalNormal` as the guide to keep the optimization well-conditioned (like Model 2), and SVI with 2000 steps per scaling step. Predictions average over 1000 posterior samples, and the same `evaluate(...)` function scores the held-out test set.

In [47]:
def heteroscedastic_model_pyro(X, y=None):
    """Heteroscedastic Bayesian linear regression in Pyro.
    
    Both mean and variance are linear functions of X.
    - Mean: mu(X) = alpha_mu + X @ beta_mu
    - Log-variance: log_sigma2(X) = alpha_v + X @ beta_v
    
    All priors are Normal(0, 1) scaled to standardized data.
    """
    n, d = X.shape
    
    # Priors for mean function
    alpha_mu = pyro.sample("alpha_mu", dist.Normal(torch.tensor(0.0), torch.tensor(1.0)))
    beta_mu = pyro.sample(
        "beta_mu",
        dist.Normal(torch.zeros(d), torch.ones(d)).to_event(1)
    )
    
    # Priors for log-variance function (allowing noise to depend on X)
    alpha_v = pyro.sample("alpha_v", dist.Normal(torch.tensor(0.0), torch.tensor(1.0)))
    beta_v = pyro.sample(
        "beta_v",
        dist.Normal(torch.zeros(d), torch.ones(d)).to_event(1)
    )
    
    # Heteroscedastic likelihood: variance depends on X
    mean = alpha_mu + X @ beta_mu
    log_sigma2 = alpha_v + X @ beta_v
    sigma = torch.exp(0.5 * log_sigma2)  # convert log-variance to std deviation
    
    with pyro.plate("data", n):
        return pyro.sample("y", dist.Normal(mean, sigma), obs=y)


# Run heteroscedastic SVI across the scaling grid
rows_hetero = []

for n_cells in SCALING_GRID:
    s = slice_step(n_cells)
    
    # Convert to torch tensors
    X_train_torch = torch.tensor(s["X_train"], dtype=torch.float32)
    y_train_torch = torch.tensor(s["y_train"], dtype=torch.float32)
    X_test_torch = torch.tensor(s["X_test"], dtype=torch.float32)
    
    # Setup SVI
    pyro.clear_param_store()
    pyro.set_rng_seed(SEED)
    torch.manual_seed(SEED)
    
    guide = AutoDiagonalNormal(heteroscedastic_model_pyro)
    optimizer = Adam({"lr": 0.01})
    svi = SVI(heteroscedastic_model_pyro, guide, optimizer, loss=Trace_ELBO())
    
    # Training loop: optimize ELBO
    n_steps = 2000
    elbos = []
    for step in range(n_steps):
        elbo = svi.step(X_train_torch, y_train_torch)
        elbos.append(elbo)
        if step % 400 == 0:
            print(f"  [{n_cells}] step {step:4d}: ELBO = {elbo:10.2f}", end="\r")
    print(f"  [{n_cells}] completed {n_steps} steps; final ELBO = {elbos[-1]:10.2f}     ")
    
    # Posterior predictions: sample parameters from guide, compute mean predictions
    with torch.no_grad():
        n_samples = 1000
        y_pred_samples = []
        for _ in range(n_samples):
            # Sample parameters from the guide (posterior approximation)
            guide_trace = pyro.poutine.trace(guide).get_trace(X_train_torch, y_train_torch)
            alpha_mu_sample = guide_trace.nodes["alpha_mu"]["value"]
            beta_mu_sample = guide_trace.nodes["beta_mu"]["value"]
            # Compute mean prediction (using posterior mean function, ignoring variance samples)
            y_pred = alpha_mu_sample + X_test_torch @ beta_mu_sample
            y_pred_samples.append(y_pred)
        
        # Average predictions across posterior samples
        y_hat_std = torch.stack(y_pred_samples).mean(dim=0).numpy()
    
    # Back to original units (m³/ha)
    y_hat = y_hat_std * s["y_std"] + s["y_mean"]
    y_true = s["y_test"] * s["y_std"] + s["y_mean"]
    
    # Evaluate
    m = evaluate(y_true, y_hat, coords=s["coords_test"])
    m = {"n_cells": n_cells, "n_train": s["n_train"], "n_test": s["n_test"], 
         "final_elbo": float(elbos[-1]), **m}
    rows_hetero.append(m)

print("\nHeteroscedastic Results across scaling grid:")
pd.DataFrame(rows_hetero)
elbos_hetero = list(elbos)  # save for convergence plot

  [5] completed 2000 steps; final ELBO =    3321.67     
  [25] completed 2000 steps; final ELBO =   16495.60     
  [50] completed 2000 steps; final ELBO =   31854.03     
  [all] completed 2000 steps; final ELBO =   56687.87     

Heteroscedastic Results across scaling grid:


,n_cells,n_train,n_test,final_elbo,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,5,3411,10889,3321.667500,36.351221,24.895315,-1.443973,0.284209,0.535592,0.214754,0.001
1,25,14945,10889,16495.601257,36.452980,25.193790,0.485325,0.280196,0.535452,0.220913,0.001
2,50,28925,10889,31854.034701,35.341235,24.857023,0.413578,0.323432,0.568905,0.212179,0.001
3,all,52184,10889,56687.870680,35.330007,24.906366,0.820513,0.323862,0.569486,0.209355,0.001


**Reading the output.** Compare the heteroscedastic RMSE, MAE, and R² to Models 1–3 (which use constant variance). If heteroscedasticity is real in the data, the heteroscedastic model should: (i) have a competitive or better RMSE (if it's just fitting noise, RMSE stays the same or degrades); (ii) produce a lower final ELBO than homoscedastic models on the same training set (more parameters, more flexibility to explain data); (iii) show residuals with different spread at different feature values — check this visually by plotting $|\hat{r}_i|$ vs. predicted $\hat{y}_i$ or vs. specific covariates like $\text{volume}_t1$. The `alpha_v` and `beta_v` posterior samples encode which features drive variance: large posterior means for some `beta_v` components suggest those covariates strongly modulate noise. If `beta_v` is close to zero (posterior heavily peaked near 0), the data may not support heteroscedastic structure and the simpler constant-variance models are favored.

## 8. Investigating Spatial Dependence

### 8.1 Spatial feature lag covariates: capturing local context

**Why spatial lags matter.**  The Bayesian linear models above achieve strong predictive performance but leave residual Moran's I ≈ 0.18 — evidence of unmodelled spatial structure. The models know each pixel's own forest state but nothing about its neighbourhood. If surrounding pixels are taller and denser, that context shapes the growth trajectory of the pixel itself, independently of its own features.

**Approach.**  For each pixel we augment the feature vector with the mean value of each predictor across a spatial neighbourhood, giving the model access to local context without leaving the linear-in-parameters framework. The augmented matrix $[\mathbf{X},\,\bar{\mathbf{X}}_{\text{lag}}]$ is still linear, so exact Bayesian inference remains tractable.

**Neighbourhood sweep.**  Rather than committing to one neighbourhood shape, we compare eight configurations across two geometry families and one distance-based family:

| Type | Sizes tested | Description |
|------|-------------|-------------|
| Rook | radius 1, 2 | Up/down/left/right only; Manhattan distance $\leq r$ |
| Queen | radius 1, 2 | All 8 directions; Chebyshev distance $\leq r$ |
| kNN | 4, 8, 16, 32 | $k$ nearest neighbours by Euclidean distance |

The best neighbourhood configuration is selected by test-set R² at each scaling step, giving a data-driven answer to how large and what shape the relevant local context is.

**Why spatial lag covariates rather than SAR.**  The SAR model (next cell) couples observations through the *response* lag $Wy$, which breaks the conjugate-posterior structure and requires variational or MCMC inference. Spatial lag *covariates* keep the model conjugate and linear, enabling exact posterior diagnostics as a diagnostic baseline before introducing the added complexity of SAR or spatial random effects.

In [ ]:
# Step 1: Prepare spatial lag computation using BASE_COLS only (not the CLR-transformed species)
spatial_base_cols = BASE_COLS
df_spatial = df[["x", "y"] + spatial_base_cols].copy()
df_spatial[["x", "y"]] = df_spatial[["x", "y"]].apply(pd.to_numeric, errors="coerce")
df_spatial[spatial_base_cols] = df_spatial[spatial_base_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
df_spatial[["x", "y"]] = df_spatial[["x", "y"]].round(6)

# Step 2: Infer grid spacing from coordinate differences (used for rook / queen offsets)
x_vals = np.sort(df_spatial["x"].unique())
y_vals = np.sort(df_spatial["y"].unique())
x_step = np.diff(x_vals)
y_step = np.diff(y_vals)
step_candidates = np.concatenate([x_step[x_step > 0], y_step[y_step > 0]]) if (len(x_step) or len(y_step)) else np.array([])
if step_candidates.size == 0:
    raise ValueError("Could not infer a positive grid spacing from x/y coordinates.")
cell_step = float(np.min(step_candidates))

# Neighbourhood configurations to test
neighbor_configs = [
    ("rook", 1),
    ("rook", 2),
    ("queen", 1),
    ("queen", 2),
    ("knn", 4),
    ("knn", 8),
    ("knn", 16),
    ("knn", 32),
]

from sklearn.neighbors import NearestNeighbors

def compute_offsets(radius: int, mode: str) -> list[tuple[float, float]]:
    steps = range(-radius, radius + 1)
    offsets = []
    for dx_idx in steps:
        for dy_idx in steps:
            if dx_idx == 0 and dy_idx == 0:
                continue
            if mode == "rook":
                if abs(dx_idx) + abs(dy_idx) <= radius:
                    offsets.append((dx_idx * cell_step, dy_idx * cell_step))
            elif mode == "queen":
                if max(abs(dx_idx), abs(dy_idx)) <= radius:
                    offsets.append((dx_idx * cell_step, dy_idx * cell_step))
    return offsets

def lag_from_offsets(df_spatial: pd.DataFrame, offsets: list[tuple[float, float]], cols: list[str]) -> np.ndarray:
    coord_df = df_spatial[["x", "y"]].copy()
    lag_sum = np.zeros((len(df_spatial), len(cols)), dtype=float)
    lag_count = np.zeros(len(df_spatial), dtype=float)
    for dx, dy in offsets:
        neighbor = df_spatial[["x", "y"] + cols].copy()
        neighbor["x"] = (neighbor["x"] - dx).round(6)
        neighbor["y"] = (neighbor["y"] - dy).round(6)
        neighbor = neighbor.rename(columns={c: f"{c}_nbr" for c in cols})
        merged = coord_df.merge(neighbor, on=["x", "y"], how="left")
        neighbor_cols = [f"{c}_nbr" for c in cols]
        available = merged[neighbor_cols[0]].notna().to_numpy()
        lag_count += available.astype(float)
        lag_sum += merged[neighbor_cols].fillna(0.0).to_numpy()
    return np.divide(lag_sum, lag_count[:, None], out=np.zeros_like(lag_sum), where=lag_count[:, None] > 0)

def lag_from_knn(features: np.ndarray, coords: np.ndarray, k: int) -> np.ndarray:
    coords = np.asarray(coords, dtype=float)
    features = np.asarray(features, dtype=float)
    n_obs = len(coords)
    if n_obs <= 1:
        return np.zeros_like(features)
    k_eff = min(int(k), n_obs - 1)
    if k_eff <= 0:
        return np.zeros_like(features)
    nn = NearestNeighbors(n_neighbors=k_eff + 1).fit(coords)
    _, idx = nn.kneighbors(coords)
    neighbours = idx[:, 1:]
    return features[neighbours].mean(axis=1)

# Run exact inference for each neighbourhood definition and size
rows_spatial2 = []
print("Running spatial-lag exact inference across neighbourhood configurations:\n")
print(f"{'n_cells':<10} {'neighbourhood':<14} {'size':<6} {'R2':<8} {'RMSE':<10} {'MoranI':<8}")
print("-" * 70)

for nb_type, nb_size in neighbor_configs:
    if nb_type in ("rook", "queen"):
        offsets = compute_offsets(nb_size, nb_type)
        lag_mean = lag_from_offsets(df_spatial, offsets, spatial_base_cols)
    else:
        features = df_spatial[spatial_base_cols].to_numpy()
        coords = df_spatial[["x", "y"]].to_numpy()
        lag_mean = lag_from_knn(features, coords, nb_size)

    X_base_for_lag = df_spatial[spatial_base_cols].to_numpy()
    X_spatial_augmented = np.hstack([X_base_for_lag, lag_mean])

    def slice_step_spatial_variant(n_cells):
        train_bk = set(get_train_bk(splits, n_cells))
        test_bk = set(splits["test_bk"])
        tr = np.isin(bk_all, list(train_bk))
        te = np.isin(bk_all, list(test_bk))
        Xtr = X_spatial_augmented[tr]
        Xte = X_spatial_augmented[te]
        ytr = y_all[tr]
        yte = y_all[te]
        coords_te = coords_all[te]
        x_mean = Xtr.mean(axis=0)
        x_std = Xtr.std(axis=0) + 1e-8
        y_mean_local = float(ytr.mean())
        y_std_local = float(ytr.std() + 1e-8)
        Xtr_s = (Xtr - x_mean) / x_std
        Xte_s = (Xte - x_mean) / x_std
        ytr_s = (ytr - y_mean_local) / y_std_local
        yte_s = (yte - y_mean_local) / y_std_local
        return {
            "X_train": Xtr_s,
            "y_train": ytr_s,
            "X_test": Xte_s,
            "y_test": yte_s,
            "coords_test": coords_te,
            "y_mean": y_mean_local,
            "y_std": y_std_local,
            "n_train": int(tr.sum()),
            "n_test": int(te.sum()),
        }

    for n_cells in SCALING_GRID:
        s = slice_step_spatial_variant(n_cells)
        exact_result = exact_bayesian_linear_regression(
            s,
            n_samples=500,
            tau=10.0,
            a0=1e-2,
            b0=1e-2,
        )
        m = exact_result["metrics"]
        row = {
            "n_cells": n_cells,
            "n_train": s["n_train"],
            "n_test": s["n_test"],
            "neighbourhood_type": nb_type,
            "neighbourhood_size": nb_size,
            **m,
        }
        rows_spatial2.append(row)
        print(f"{str(n_cells):<10} {nb_type:<14} {nb_size:<6} {m['R2']:<8.3f} {m['RMSE']:<10.2f} {m['MoranI']:<8.4f}")

results_spatial_lag_variants = pd.DataFrame(rows_spatial2)
print("\nCompleted neighbourhood sweep. Summary (best R2 per scaling step):")
if len(results_spatial_lag_variants) > 0:
    best_per_scale = results_spatial_lag_variants.loc[results_spatial_lag_variants.groupby("n_cells")["R2"].idxmax()]
    print(best_per_scale[["n_cells", "neighbourhood_type", "neighbourhood_size", "R2", "RMSE", "MoranI"]].to_string(index=False))
else:
    print("No neighbourhood variants were fitted.")

Running spatial-lag exact inference across neighbourhood configurations:

n_cells    neighbourhood  size   R2       RMSE       MoranI  
----------------------------------------------------------------------
5          rook           1      0.371    34.08      0.1832  
25         rook           1      0.377    33.90      0.1897  
50         rook           1      0.390    33.57      0.1803  
all        rook           1      0.389    33.58      0.1797  
5          rook           2      0.396    33.40      0.1758  
25         rook           2      0.382    33.77      0.1936  
50         rook           2      0.392    33.50      0.1832  
all        rook           2      0.391    33.52      0.1831  
5          queen          1      0.396    33.40      0.1757  
25         queen          1      0.375    33.97      0.1971  
50         queen          1      0.390    33.55      0.1828  
all        queen          1      0.390    33.57      0.1823  
5          queen          2      0.395    33.43  

**Reading the output.** The neighbourhood sweep tests eight configurations (rook r=1,2; queen r=1,2; kNN k=4,8,16,32). Predictive accuracy (RMSE, R²) changes only marginally across configurations and shows little improvement over plain BLR: the own-feature vector already captures most of what the neighbours can add through a linear combination. Moran's I remains ≈ 0.18 regardless of neighbourhood shape or size, confirming that augmenting the feature matrix with spatial averages does not resolve the residual autocorrelation — the spatial structure is not a simple smoothing effect but a discrete grouping effect (management units) that linear lags cannot absorb.

## Spatial autoregressive (SAR) model: beyond spatial lag covariates

**Why SAR instead of just spatial lag features.** The spatial-lag-covariate model (cell before this) captures local context by augmenting predictors with neighbor averages—a linear approach that stays within the conjugate-posterior framework. SAR takes a step further by directly modeling spatial autocorrelation in the response: $y_i = \rho W y_i + \mathbf{X}_i \beta + \varepsilon_i$, where $Wy$ is the lag of neighboring growth values. This allows the model to absorb spatial structure that can't be explained by local covariates alone.

**Trade-off: complexity for interpretation.** SAR prediction is more complex (no closed form; requires iteration or approximation), and the reduced-form posterior is harder to interpret than conjugate methods. However, when residual spatial autocorrelation remains high despite spatial lags and covariates, SAR provides a principled alternative. We fit it at a few neighborhood sizes ($k$) to explore sensitivity and compare test-set performance against the linear spatial-lag model.

In [50]:
# Bayesian SAR-style model fitted with SVI.
# This replaces the PySAL MLE fit with the same Pyro/SVI pattern used earlier in the notebook.

import numpy as np
import torch
import pyro
import pyro.distributions as dist
from pyro.contrib.autoguide import AutoDiagonalNormal
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import Adam
from sklearn.neighbors import NearestNeighbors


# SAR run toggle: run only the smallest scaling step by default (like the MCMC toggle)
# Set RUN_SAR_FULL_GRID = True to run the full SCALING_GRID.
RUN_SAR_FULL_GRID = True  # Set to True to run full scaling grid
if RUN_SAR_FULL_GRID:
    sar_scaling_steps = SCALING_GRID
else:
    sar_scaling_steps = [SCALING_GRID[0]]


# Test k-neighborhood sizes for spatial context aggregation (doubling pattern for efficiency)
k_values = [8, 16, 32, 64]
rows_sar = []


def sar_response_lag_train(coords_train: np.ndarray, y_train: np.ndarray, k: int) -> np.ndarray:
    """Wy for training: mean of k nearest *other* training neighbours' y (self excluded)."""
    coords_train = np.asarray(coords_train, dtype=float)
    y_train = np.asarray(y_train, dtype=float).ravel()
    k_eff = min(int(k), len(y_train) - 1)
    if k_eff <= 0:
        return np.zeros(len(y_train), dtype=float)
    nn = NearestNeighbors(n_neighbors=k_eff + 1).fit(coords_train)
    _, idx = nn.kneighbors(coords_train)
    return y_train[idx[:, 1:]].mean(axis=1)  # idx[:, 0] is self


def sar_response_lag_test(coords_train: np.ndarray, y_train: np.ndarray,
                          coords_test: np.ndarray, k: int) -> np.ndarray:
    """Wy for test: mean of k nearest *training* neighbours' y.

    Training y is known at prediction time and BK-disjoint splits keep test
    pixels out of training BK cells, so this is leak-free.
    """
    coords_train = np.asarray(coords_train, dtype=float)
    y_train = np.asarray(y_train, dtype=float).ravel()
    coords_test = np.asarray(coords_test, dtype=float)
    k_eff = min(int(k), len(y_train))
    if k_eff <= 0:
        return np.zeros(len(coords_test), dtype=float)
    nn = NearestNeighbors(n_neighbors=k_eff).fit(coords_train)
    _, idx = nn.kneighbors(coords_test)
    return y_train[idx].mean(axis=1)


def sar_svi_model(X, spatial_context, y=None):
    """Bayesian SAR: mean = alpha + X @ beta + rho * Wy, where Wy is the spatial lag of y."""
    n_obs, n_features = X.shape
    alpha = pyro.sample("alpha", dist.Normal(torch.tensor(0.0), torch.tensor(1.0)))
    beta = pyro.sample(
        "beta",
        dist.Normal(torch.zeros(n_features), torch.ones(n_features)).to_event(1),
    )
    rho = pyro.sample("rho", dist.Normal(torch.tensor(0.0), torch.tensor(1.0)))
    sigma = pyro.sample("sigma", dist.HalfNormal(torch.tensor(1.0)))
    mean = alpha + X @ beta + rho * spatial_context
    with pyro.plate("data", n_obs):
        return pyro.sample("y", dist.Normal(mean, sigma), obs=y)


print("SAR model comparison across scaling steps and neighborhood sizes via SVI:\n")
print(f"{'n_cells':<10} {'k':<6} {'rho':<8} {'n_train':<8} {'n_test':<8} {'R2_test':<8} {'RMSE_test':<10} {'MAE_test':<10} {'MoranI':<8}")
print("-" * 96)

for n_cells in sar_scaling_steps:
    s = slice_step(n_cells)

    # Extract train and test data at this scaling step
    X_train = s["X_train"]
    X_test = s["X_test"]
    y_train = s["y_train"]
    y_test = s["y_test"]
    coords_test = s["coords_test"]
    y_mean = s["y_mean"]
    y_std = s["y_std"]

    # Get corresponding training coordinates
    train_bk = set(get_train_bk(splits, n_cells))
    tr_idx = np.isin(bk_all, list(train_bk))
    coords_train = coords_all[tr_idx]
    n_train_obs = len(coords_train)

    for k in k_values:
        # Skip k-values that are too large for the training set
        if k >= n_train_obs:
            continue

        # True SAR response lag Wy: uses standardised training y values.
        # Training lag excludes self (W diagonal = 0). Test lag looks up into
        # the training set only — valid because BK-disjoint splits keep test
        # pixels in separate BK cells from all training points.
        train_context = sar_response_lag_train(coords_train, y_train, k=k)
        test_context = sar_response_lag_test(coords_train, y_train, coords_test, k=k)

        context_mean = float(train_context.mean())
        context_std = float(train_context.std() + 1e-8)
        train_context_s = (train_context - context_mean) / context_std
        test_context_s = (test_context - context_mean) / context_std

        # Convert to torch tensors
        X_train_torch = torch.tensor(X_train, dtype=torch.float32)
        y_train_torch = torch.tensor(y_train, dtype=torch.float32)
        X_test_torch = torch.tensor(X_test, dtype=torch.float32)
        train_context_torch = torch.tensor(train_context_s, dtype=torch.float32)
        test_context_torch = torch.tensor(test_context_s, dtype=torch.float32)

        # Setup SVI
        pyro.clear_param_store()
        pyro.set_rng_seed(SEED)
        torch.manual_seed(SEED)

        guide = AutoDiagonalNormal(sar_svi_model)
        optimizer = Adam({"lr": 0.01})
        svi = SVI(sar_svi_model, guide, optimizer, loss=Trace_ELBO())

        # Training loop: optimize ELBO
        n_steps = 2000
        elbos = []
        for step in range(n_steps):
            elbo = svi.step(X_train_torch, train_context_torch, y_train_torch)
            elbos.append(elbo)
            if step % 300 == 0:
                print(f"  [{n_cells}] k={k:<3d} step {step:4d}: ELBO = {elbo:10.2f}", end="\r")
        print(f"  [{n_cells}] k={k:<3d} completed {n_steps} steps; final ELBO = {elbos[-1]:10.2f}     ")

        # Posterior predictions: sample parameters from guide, compute mean predictions
        with torch.no_grad():
            n_samples = 1000
            y_pred_samples = []
            rho_samples = []
            for _ in range(n_samples):
                guide_trace = pyro.poutine.trace(guide).get_trace(
                    X_train_torch,
                    train_context_torch,
                    y_train_torch,
                )
                alpha_sample = guide_trace.nodes["alpha"]["value"]
                beta_sample = guide_trace.nodes["beta"]["value"]
                rho_sample = guide_trace.nodes["rho"]["value"]
                y_pred = alpha_sample + X_test_torch @ beta_sample + rho_sample * test_context_torch
                y_pred_samples.append(y_pred)
                rho_samples.append(rho_sample)

            y_hat_std = torch.stack(y_pred_samples).mean(dim=0).numpy()
            rho = float(torch.stack(rho_samples).mean().item())

        # Back to original units (m³/ha)
        y_hat = y_hat_std * y_std + y_mean
        y_true = y_test * y_std + y_mean

        # Evaluate
        m = evaluate(y_true, y_hat, coords=coords_test)
        m = {
            "n_cells": n_cells,
            "k": k,
            "rho": rho,
            "n_train": s["n_train"],
            "n_test": s["n_test"],
            "final_elbo": float(elbos[-1]),
            **m,
        }
        rows_sar.append(m)

        print(
            f"{str(n_cells):<10} {k:<6} {rho:>7.3f}  {s['n_train']:<8} {s['n_test']:<8} "
            f"{m['R2']:>7.3f}  {m['RMSE']:>9.2f}  {m['MAE']:>9.2f}  {m['MoranI']:>7.4f}"
        )

results_sar = pd.DataFrame(rows_sar)
print("\n" + "=" * 96)
if len(rows_sar) > 0:
    print(f"Fitted {len(rows_sar)} SAR-style SVI models with k ∈ {k_values}.")
    print(f"\nBest k per scaling step (highest R²):")

    best_per_scale = results_sar.loc[results_sar.groupby("n_cells")["R2"].idxmax()]
    print(best_per_scale[["n_cells", "k", "rho", "R2", "RMSE", "MoranI", "final_elbo"]].to_string(index=False))
else:
    print("No SAR models were successfully fitted.")
elbos_sar = list(elbos)  # save for convergence plot

SAR model comparison across scaling steps and neighborhood sizes via SVI:

n_cells    k      rho      n_train  n_test   R2_test  RMSE_test  MAE_test   MoranI  
------------------------------------------------------------------------------------------------
  [5] k=8   completed 2000 steps; final ELBO =    3716.61     
5          8        0.390  3411     10889      0.129      40.11      31.43   0.4131
  [5] k=16  completed 2000 steps; final ELBO =    3723.74     
5          16       0.394  3411     10889      0.114      40.44      32.06   0.4109
  [5] k=32  completed 2000 steps; final ELBO =    3788.95     
5          32       0.337  3411     10889      0.173      39.07      30.56   0.3740
  [5] k=64  completed 2000 steps; final ELBO =    3852.41     
5          64       0.257  3411     10889      0.252      37.15      28.34   0.3193
  [25] k=8   completed 2000 steps; final ELBO =   16837.76     
25         8        0.382  14945    10889      0.227      37.77      27.63   0.3299
  [25] 

**Reading the output.** Moran's I increases to ≈ 0.30–0.40, substantially *worse* than the plain BLR baseline (0.18). This is the BK-disjoint problem in action: training pixels compute their response lag $Wy$ from within-BK neighbours, but test pixels must pull their lag from training BK cells across the 500 m boundary. The learned $\rho$ is calibrated for short within-BK lags and is mis-specified at test time, introducing correlated prediction errors that inflate Moran's I. The declining Moran's I with more training BK cells (larger scaling steps) reflects the expanding training neighbourhood reducing — but never eliminating — this cross-boundary mismatch. The low $\rho$ values (≈ 0.05) suggest the data do not support a contagion mechanism anyway: spatial correlation here comes from *shared latent structure* (management units), not from neighbours causing each other's growth.

### Model 5. Feature-based Cluster Intercept Model

**Why spatial models fail on this dataset.**  Both the SAR model and the sparse GP try to transfer spatial information across BK cell boundaries, but the train/test split is *BK-cell-disjoint*: the 24 held-out test cells share no spatial neighbours with the training set.  Any model that relies on proximity — contagion-style response lags (SAR) or a smooth covariance kernel (GP) — must therefore extrapolate spatially rather than interpolate.  The worsening Moran's I confirms this (SAR: 0.3–0.4; GP: 0.42; plain BLR: 0.18).

**The key insight.**  The residual spatial correlation is not caused by *contagion* but by *shared latent management structure*: pixels in the same management unit share stand history, thinning regime, and species mix, driving correlated growth beyond what the observed covariates already capture.  Management units are spatially compact, but their *signatures live in the feature vector* **x**.  If we cluster pixels by features rather than location, each cluster approximates a management-unit type, and that identity transfers to unseen test cells via feature similarity — no spatial proximity required.

**Model.**  For pixel $i$ assigned to cluster $k(i)$ (k-means on the standardised feature matrix $\mathbf{X}$):

$$
y_i = \alpha_{k(i)} + \mathbf{x}_i^\top \boldsymbol{\beta} + \varepsilon_i,\quad \varepsilon_i \sim \mathcal{N}(0,\,\sigma^2)
$$

with hierarchical priors:

$$
\alpha_k \sim \mathcal{N}(0,\,\sigma_\alpha),\quad \sigma_\alpha \sim \text{HalfNormal}(1),\quad \boldsymbol{\beta} \sim \mathcal{N}(\mathbf{0},\,\mathbf{I}),\quad \sigma \sim \text{HalfNormal}(1).
$$

**Transfer to new cells.**  At test time each pixel is assigned to the nearest cluster centroid in feature space.  Because centroids represent *types of forest stand* rather than locations, the learned intercepts transfer to unseen BK cells as long as those cells contain recognisable stand types — a far weaker assumption than spatial smoothness.

In [51]:
# ============================================================
# Feature-based Cluster Intercept Model (Pyro SVI)
# k-means clusters on feature space proxy for management units;
# random intercept per cluster transfers via feature similarity.
# ============================================================
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.contrib.autoguide import AutoDiagonalNormal
from pyro.optim import Adam
from sklearn.cluster import KMeans

K_CLUSTERS = 20    # latent stand-type clusters
N_STEPS_CL = 2000
LR_CL      = 0.01
SEED_CL    = 42

rows_cluster = []


def cluster_intercept_model(X, cluster_ids, K, y=None):
    """y_i = alpha[k(i)] + x_i @ beta + eps_i,  alpha_k ~ N(0, sigma_a)."""
    n, d = X.shape
    sigma_a = pyro.sample("sigma_a", dist.HalfNormal(torch.tensor(1.0)))
    alpha   = pyro.sample(
        "alpha",
        dist.Normal(torch.zeros(K), sigma_a * torch.ones(K)).to_event(1),
    )
    beta  = pyro.sample("beta",  dist.Normal(torch.zeros(d), torch.ones(d)).to_event(1))
    sigma = pyro.sample("sigma", dist.HalfNormal(torch.tensor(1.0)))
    mean  = alpha[cluster_ids] + X @ beta
    with pyro.plate("data", n):
        return pyro.sample("y", dist.Normal(mean, sigma), obs=y)


print(f"Cluster Intercept | k-means K={K_CLUSTERS} on features | Pyro SVI")
print(f"Random intercepts per cluster; shared beta across all clusters")
print()
print(f"{'n_cells':<10} {'sigma_a':>10} {'n_train':<8} {'n_test':<8}"
      f" {'R2':>8} {'RMSE':>10} {'MoranI':>8}")
print("-" * 82)

for n_cells in SCALING_GRID:
    pyro.clear_param_store()
    pyro.set_rng_seed(SEED_CL)
    torch.manual_seed(SEED_CL)

    s = slice_step(n_cells)
    y_mean, y_std = s["y_mean"], s["y_std"]

    # cluster pixels by features (already standardised by slice_step)
    km = KMeans(n_clusters=K_CLUSTERS, random_state=SEED_CL, n_init=5)
    km.fit(s["X_train"])
    ids_train = km.predict(s["X_train"])
    ids_test  = km.predict(s["X_test"])

    X_train_t   = torch.tensor(s["X_train"], dtype=torch.float32)
    y_train_t   = torch.tensor(s["y_train"], dtype=torch.float32)
    X_test_t    = torch.tensor(s["X_test"],  dtype=torch.float32)
    ids_train_t = torch.tensor(ids_train,    dtype=torch.long)
    ids_test_t  = torch.tensor(ids_test,     dtype=torch.long)

    guide     = AutoDiagonalNormal(cluster_intercept_model)
    optimizer = Adam({"lr": LR_CL})
    svi       = SVI(cluster_intercept_model, guide, optimizer, loss=Trace_ELBO())

    elbos = []
    for step in range(N_STEPS_CL):
        elbo = svi.step(X_train_t, ids_train_t, K_CLUSTERS, y_train_t)
        elbos.append(elbo)
        if step % 400 == 0:
            print(f"  [{n_cells}] step {step:4d}: ELBO = {elbo:10.2f}", end="\r")
    print(f"  [{n_cells}] done {N_STEPS_CL} steps; final ELBO = {elbos[-1]:10.2f}     ")

    with torch.no_grad():
        y_pred_samples = []
        for _ in range(1000):
            guide_trace = pyro.poutine.trace(guide).get_trace(
                X_train_t, ids_train_t, K_CLUSTERS, y_train_t
            )
            alpha_s = guide_trace.nodes["alpha"]["value"]  # (K,)
            beta_s  = guide_trace.nodes["beta"]["value"]   # (d,)
            y_pred_samples.append(alpha_s[ids_test_t] + X_test_t @ beta_s)
        y_hat_std = torch.stack(y_pred_samples).mean(dim=0).numpy()

        # posterior sigma_a for diagnostics
        g_trace = pyro.poutine.trace(guide).get_trace(
            X_train_t, ids_train_t, K_CLUSTERS, y_train_t
        )
        sigma_a_post = float(g_trace.nodes["sigma_a"]["value"].item())

    y_hat  = y_hat_std   * y_std + y_mean
    y_true = s["y_test"] * y_std + y_mean

    m   = evaluate(y_true, y_hat, coords=s["coords_test"])
    row = dict(n_cells=n_cells, n_train=s["n_train"], n_test=s["n_test"],
               K=K_CLUSTERS, sigma_a=sigma_a_post,
               final_elbo=float(elbos[-1]), **m)
    rows_cluster.append(row)

    print(f"{str(n_cells):<10} {sigma_a_post:>10.4f} {s['n_train']:<8} {s['n_test']:<8}"
          f" {m['R2']:>8.4f} {m['RMSE']:>10.2f} {m['MoranI']:>8.4f}")

results_cluster = pd.DataFrame(rows_cluster)
print()
print("sigma_a > 0 means clusters have meaningfully different intercepts.")
print("Compare MoranI to BLR baseline (0.18) to assess whether spatial residuals improved.")

elbos_cluster = list(elbos)  # save for convergence plot

Cluster Intercept | k-means K=20 on features | Pyro SVI
Random intercepts per cluster; shared beta across all clusters

n_cells       sigma_a n_train  n_test         R2       RMSE   MoranI
----------------------------------------------------------------------------------
  [5] done 2000 steps; final ELBO =    3904.09     
5              0.1652 3411     10889      0.3748      33.97   0.1723
  [25] done 2000 steps; final ELBO =   17706.01     
25             0.1494 14945    10889      0.3647      34.25   0.1812
  [50] done 2000 steps; final ELBO =   33949.78     
50             0.2367 28925    10889      0.3808      33.81   0.1776
  [all] done 2000 steps; final ELBO =   60854.54     
all            0.2467 52184    10889      0.3882      33.61   0.1715

sigma_a > 0 means clusters have meaningfully different intercepts.
Compare MoranI to BLR baseline (0.18) to assess whether spatial residuals improved.


**Reading the output.** `sigma_a` ≈ 0.16–0.24 (standardised units) confirms that the 20 cluster intercepts genuinely differ from each other — collapsing to zero would mean all clusters grow identically after accounting for features, which is not the case. Multiplied by `y_std` (≈ 50 m³/ha), the between-cluster spread is roughly ±10 m³/ha, a meaningful management-unit effect. Moran's I drops to ≈ 0.17 (from 0.18 for plain BLR), a modest but real improvement: feature-space clusters proxy management-unit types, and their learned intercepts transfer cleanly to unseen BK cells via feature similarity rather than spatial proximity — avoiding the BK-disjoint extrapolation problem that worsened SAR and GP.

#### Parameter tuning: number of clusters K

K is the only free parameter in Model 5. It controls how finely the feature space is partitioned into stand types, and therefore how many distinct management-unit intercepts the model can represent.

**Trade-off.** Too few clusters → coarse groupings that behave like plain BLR. Too many → each cluster has insufficient training points to estimate a reliable intercept, and the model overfits: `sigma_a` inflates, cluster intercepts absorb noise, RMSE rises. The collapse seen at large K with small training sets (high RMSE alongside artificially low Moran's I from spatially random errors) illustrates this directly.

Run the cell below across the candidate values, inspect the plot, and copy the best-performing K into the `K_CLUSTERS` constant in the single-run cell above.

In [63]:
# ============================================================
# K sweep for Model 5: Cluster Intercept
# Runs the same model for a range of K values to find how many
# feature-space clusters the data actually support.
# ============================================================
from sklearn.cluster import KMeans
import torch, pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.contrib.autoguide import AutoDiagonalNormal
from pyro.optim import Adam

K_SWEEP_VALUES  = [100, 200, 500, 1000, 2000]
N_STEPS_KSWEEP  = 2000
SEED_KSWEEP     = 42

rows_k_sweep = []

print(f"K sweep | Model 5 Cluster Intercept | K ∈ {K_SWEEP_VALUES}")
print(f"{'K':<6} {'n_cells':<10} {'sigma_a':>10} {'R2':>8} {'RMSE':>10} {'MoranI':>8}")
print("-" * 60)

for K in K_SWEEP_VALUES:
    for n_cells in SCALING_GRID:
        pyro.clear_param_store()
        pyro.set_rng_seed(SEED_KSWEEP)
        torch.manual_seed(SEED_KSWEEP)

        s = slice_step(n_cells)
        y_mean, y_std = s["y_mean"], s["y_std"]

        km = KMeans(n_clusters=K, random_state=SEED_KSWEEP, n_init=5)
        km.fit(s["X_train"])
        ids_train = km.predict(s["X_train"])
        ids_test  = km.predict(s["X_test"])

        X_tr_t   = torch.tensor(s["X_train"],  dtype=torch.float32)
        y_tr_t   = torch.tensor(s["y_train"],  dtype=torch.float32)
        X_te_t   = torch.tensor(s["X_test"],   dtype=torch.float32)
        ids_tr_t = torch.tensor(ids_train,     dtype=torch.long)
        ids_te_t = torch.tensor(ids_test,      dtype=torch.long)

        guide     = AutoDiagonalNormal(cluster_intercept_model)
        optimizer = Adam({"lr": 0.01})
        svi       = SVI(cluster_intercept_model, guide, optimizer, loss=Trace_ELBO())

        for step in range(N_STEPS_KSWEEP):
            svi.step(X_tr_t, ids_tr_t, K, y_tr_t)

        with torch.no_grad():
            y_pred_samples = []
            for _ in range(500):
                g = pyro.poutine.trace(guide).get_trace(
                    X_tr_t, ids_tr_t, K, y_tr_t
                )
                alpha_s = g.nodes["alpha"]["value"]
                beta_s  = g.nodes["beta"]["value"]
                y_pred_samples.append(alpha_s[ids_te_t] + X_te_t @ beta_s)
            y_hat_std = torch.stack(y_pred_samples).mean(0).numpy()

            g = pyro.poutine.trace(guide).get_trace(X_tr_t, ids_tr_t, K, y_tr_t)
            sigma_a_post = float(g.nodes["sigma_a"]["value"].item())

        y_hat  = y_hat_std    * y_std + y_mean
        y_true = s["y_test"]  * y_std + y_mean

        m = evaluate(y_true, y_hat, coords=s["coords_test"])
        row = dict(K=K, n_cells=n_cells, n_train=s["n_train"], n_test=s["n_test"],
                   sigma_a=sigma_a_post, **m)
        rows_k_sweep.append(row)
        print(f"{K:<6} {str(n_cells):<10} {sigma_a_post:>10.4f}"
              f" {m['R2']:>8.4f} {m['RMSE']:>10.2f} {m['MoranI']:>8.4f}")

results_k_sweep = pd.DataFrame(rows_k_sweep)

# Best K per scaling step (lowest RMSE)
best_k_per_step = results_k_sweep.loc[
    results_k_sweep.groupby("n_cells")["RMSE"].idxmin(),
    ["n_cells", "K", "RMSE", "MoranI"]
]
K_best = int(results_k_sweep.groupby("K")["RMSE"].mean().idxmin())
rows_cluster_best = results_k_sweep[results_k_sweep["K"] == K_best].to_dict("records")

print(f"\nBest K per scaling step:")
print(best_k_per_step.to_string(index=False))
print(f"\nOverall best K (lowest mean RMSE): K = {K_best}")


K sweep | Model 5 Cluster Intercept | K ∈ [100, 200, 500, 1000, 2000]
K      n_cells       sigma_a       R2       RMSE   MoranI
------------------------------------------------------------
100    5              0.2734   0.3490      34.67   0.1747
100    25             0.4190   0.3666      34.20   0.1690
100    50             0.4270   0.3664      34.20   0.1698
100    all            0.4067   0.3754      33.96   0.1653
200    5              0.2382   0.3908      33.54   0.1564
200    25             0.2978   0.3899      33.56   0.1577
200    50             0.3467   0.4005      33.27   0.1562
200    all            0.3653   0.4137      32.90   0.1473
500    5              0.3944   0.3594      34.39   0.1588
500    25             0.3158   0.3935      33.46   0.1573
500    50             0.4231   0.4074      33.08   0.1563
500    all            0.4560   0.4198      32.73   0.1476
1000   5              0.4821   0.3429      34.83   0.1475
1000   25             0.3206   0.3932      33.47   0.1537

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(SCALING_GRID)))

for color, n_cells in zip(colors, SCALING_GRID):
    sub = results_k_sweep[results_k_sweep["n_cells"] == n_cells].sort_values("K")
    ax1.plot(sub["K"], sub["RMSE"],   marker="o", color=color, label=f"n_cells={n_cells}")
    ax2.plot(sub["K"], sub["MoranI"], marker="o", color=color, label=f"n_cells={n_cells}")

# BLR baseline on Moran's I
ax2.axhline(0.18, color="black", linestyle="--", linewidth=1, label="BLR baseline (0.18)")

for ax, ylabel, title in [
    (ax1, "RMSE  (m³/ha)",  "Test RMSE vs number of clusters K"),
    (ax2, "Moran's I",      "Residual Moran's I vs number of clusters K"),
]:
    ax.set_xlabel("K  (number of clusters)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(K_SWEEP_VALUES)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%d"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    f"Model 5 — Cluster Intercept: sensitivity to K  "
    f"(best overall K = {K_best})",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()


**Reading the K sweep plot.** Look for an elbow in RMSE — the point where adding more clusters stops reducing prediction error. Moran's I should fall alongside RMSE while the model is capturing genuine spatial structure; if Moran's I keeps falling while RMSE rises, the model is overfitting cluster intercepts (random prediction errors appear spatially uncorrelated, masking the deterioration). The optimal K sits just left of where RMSE begins to increase.

### Model 6. Cluster Intercept + Spatial Feature Lag

**What the cluster model still misses.**  The cluster intercept model (Model 5) captures the management-unit *type* of each pixel but ignores its *position within the local feature landscape*. A pixel that is shorter than all its neighbours is probably a recently thinned gap inside a mature unit and will grow fast; a pixel of identical features surrounded by equally short neighbours is just an average stand. Same observed features, different local context, different expected growth.

**Spatial feature lag — truly local.**  For each pixel $i$ we compute the mean standardised feature vector of its $k$ nearest spatial neighbours:

$$
W\mathbf{x}_i = \frac{1}{k}\sum_{j \in \mathcal{N}(i)} \mathbf{x}_j
$$

Crucially, $\mathbf{x}$ is observed for **every** pixel in `X_all`, so the neighbour index is built on `coords_all` — the full study area, not just the training BK cells. A test pixel's $W\mathbf{x}$ is the mean feature of its *true* local neighbours (which may themselves be test pixels), not a proxy pulled from distant training areas. This is what breaks the BK-disjoint problem for the spatial component: the relationship $\boldsymbol{\gamma}$ is learned from training data, but it is applied at test time to the genuine local context of each test pixel.

**Combined model.**

$$
y_i = \alpha_{k(i)} + \mathbf{x}_i^\top\boldsymbol{\beta} + W\mathbf{x}_i^\top\boldsymbol{\gamma} + \varepsilon_i
$$

$$
\alpha_k \sim \mathcal{N}(0,\,\sigma_\alpha),\quad \sigma_\alpha \sim \text{HalfNormal}(1),\quad \boldsymbol{\beta},\boldsymbol{\gamma} \sim \mathcal{N}(\mathbf{0},\,\mathbf{I}),\quad \sigma \sim \text{HalfNormal}(1).
$$

The cluster intercepts absorb management-unit type shifts; $\boldsymbol{\gamma}$ encodes how the local feature contrast (am I taller or shorter than my neighbourhood?) predicts growth. If $|\boldsymbol{\gamma}| > 0$ and Moran's I falls below the cluster-only baseline, the local context is carrying genuine signal.

In [66]:
# ============================================================
# Model 6: Cluster Intercept + Spatial Feature Lag (Pyro SVI)
# Wx is computed from ALL pixels (train + test) using coords_all.
# Features are observed everywhere so no leakage; each pixel
# gets its TRUE local neighbour context, not a cross-BK proxy.
# ============================================================
import torch
import numpy as np
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.contrib.autoguide import AutoDiagonalNormal
from pyro.optim import Adam
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

K_CFL       = 500    # clusters (same as Model 5)
K_SPATIAL   = 16   # spatial neighbours for feature lag
N_STEPS_CFL = 2000
LR_CFL      = 0.01
SEED_CFL    = 42

rows_cfl = []


def compute_WX(coords_ref, X_ref, coords_query, k, exclude_self=False):
    """Mean standardised feature of k nearest neighbours from coords_ref.

    Set exclude_self=True when query pixels are contained in coords_ref (self-lag).
    """
    k_eff = min(k + int(exclude_self), len(X_ref))
    nn = NearestNeighbors(n_neighbors=k_eff).fit(coords_ref)
    _, idx = nn.kneighbors(coords_query)
    if exclude_self:
        idx = idx[:, 1:]   # drop column 0 (self)
    return X_ref[idx].mean(axis=1)   # (n_query, d)


def cluster_featurelag_model(X, WX, cluster_ids, K, y=None):
    """y_i = alpha[k(i)] + x_i @ beta + Wx_i @ gamma + eps_i."""
    n, d = X.shape
    sigma_a = pyro.sample("sigma_a", dist.HalfNormal(torch.tensor(1.0)))
    alpha   = pyro.sample(
        "alpha",
        dist.Normal(torch.zeros(K), sigma_a * torch.ones(K)).to_event(1),
    )
    beta  = pyro.sample("beta",  dist.Normal(torch.zeros(d), torch.ones(d)).to_event(1))
    gamma = pyro.sample("gamma", dist.Normal(torch.zeros(d), torch.ones(d)).to_event(1))
    sigma = pyro.sample("sigma", dist.HalfNormal(torch.tensor(1.0)))
    mean  = alpha[cluster_ids] + X @ beta + WX @ gamma
    with pyro.plate("data", n):
        return pyro.sample("y", dist.Normal(mean, sigma), obs=y)


print(f"Cluster + Feature Lag | K={K_CFL} clusters | k={K_SPATIAL} true local neighbours")
print(f"Wx built from all pixels (coords_all / X_all): each pixel sees its real neighbours")
print(f"y_i = alpha_k + x_i @ beta + Wx_i @ gamma + eps")
print()
print(f"{'n_cells':<10} {'sigma_a':>10} {'|gamma|':>8} {'n_train':<8} {'n_test':<8}"
      f" {'R2':>8} {'RMSE':>10} {'MoranI':>8}")
print("-" * 90)

for n_cells in SCALING_GRID:
    pyro.clear_param_store()
    pyro.set_rng_seed(SEED_CFL)
    torch.manual_seed(SEED_CFL)

    s = slice_step(n_cells)
    y_mean, y_std = s["y_mean"], s["y_std"]

    train_bk     = set(get_train_bk(splits, n_cells))
    tr_idx       = np.isin(bk_all, list(train_bk))
    coords_train = coords_all[tr_idx]
    coords_test  = s["coords_test"]

    X_train_s = s["X_train"]   # standardised with training mean/std
    X_test_s  = s["X_test"]

    # ---- standardise ALL pixels with training statistics -----------------
    # (same mean/std that slice_step used, so X_all_s[tr_idx] == X_train_s)
    x_mean = X_all[tr_idx].mean(axis=0)
    x_std  = X_all[tr_idx].std(axis=0) + 1e-8
    X_all_s = (X_all - x_mean) / x_std

    # ---- TRUE local feature lags: each pixel uses its actual neighbours --
    # coords_all covers every pixel; X_all_s is feature-observable everywhere.
    # No BK boundary restriction: a test pixel's neighbours are whoever is
    # spatially close — train or test — exactly as intended.
    WX_train = compute_WX(coords_all, X_all_s, coords_train,
                          k=K_SPATIAL, exclude_self=True)
    WX_test  = compute_WX(coords_all, X_all_s, coords_test,
                          k=K_SPATIAL, exclude_self=False)

    # ---- cluster assignments on own features -----------------------------
    km = KMeans(n_clusters=K_CFL, random_state=SEED_CFL, n_init=5)
    km.fit(X_train_s)
    ids_train = km.predict(X_train_s)
    ids_test  = km.predict(X_test_s)

    # ---- torch tensors ---------------------------------------------------
    X_tr_t   = torch.tensor(X_train_s,      dtype=torch.float32)
    WX_tr_t  = torch.tensor(WX_train,       dtype=torch.float32)
    y_tr_t   = torch.tensor(s["y_train"],   dtype=torch.float32)
    X_te_t   = torch.tensor(X_test_s,       dtype=torch.float32)
    WX_te_t  = torch.tensor(WX_test,        dtype=torch.float32)
    ids_tr_t = torch.tensor(ids_train,      dtype=torch.long)
    ids_te_t = torch.tensor(ids_test,       dtype=torch.long)

    # ---- SVI -------------------------------------------------------------
    guide     = AutoDiagonalNormal(cluster_featurelag_model)
    optimizer = Adam({"lr": LR_CFL})
    svi       = SVI(cluster_featurelag_model, guide, optimizer, loss=Trace_ELBO())

    elbos = []
    for step in range(N_STEPS_CFL):
        elbo = svi.step(X_tr_t, WX_tr_t, ids_tr_t, K_CFL, y_tr_t)
        elbos.append(elbo)
        if step % 400 == 0:
            print(f"  [{n_cells}] step {step:4d}: ELBO = {elbo:10.2f}", end="\r")
    print(f"  [{n_cells}] done {N_STEPS_CFL} steps; final ELBO = {elbos[-1]:10.2f}     ")

    # ---- posterior predictive mean ---------------------------------------
    with torch.no_grad():
        y_pred_samples = []
        for _ in range(1000):
            g = pyro.poutine.trace(guide).get_trace(
                X_tr_t, WX_tr_t, ids_tr_t, K_CFL, y_tr_t
            )
            alpha_s = g.nodes["alpha"]["value"]   # (K,)
            beta_s  = g.nodes["beta"]["value"]    # (d,)
            gamma_s = g.nodes["gamma"]["value"]   # (d,)
            y_pred_samples.append(
                alpha_s[ids_te_t] + X_te_t @ beta_s + WX_te_t @ gamma_s
            )
        y_hat_std = torch.stack(y_pred_samples).mean(dim=0).numpy()

        g = pyro.poutine.trace(guide).get_trace(
            X_tr_t, WX_tr_t, ids_tr_t, K_CFL, y_tr_t
        )
        sigma_a_post = float(g.nodes["sigma_a"]["value"].item())
        gamma_norm   = float(g.nodes["gamma"]["value"].norm().item())

    y_hat  = y_hat_std   * y_std + y_mean
    y_true = s["y_test"] * y_std + y_mean

    m   = evaluate(y_true, y_hat, coords=coords_test)
    row = dict(n_cells=n_cells, n_train=s["n_train"], n_test=s["n_test"],
               K=K_CFL, k_spatial=K_SPATIAL,
               sigma_a=sigma_a_post, gamma_norm=gamma_norm,
               final_elbo=float(elbos[-1]), **m)
    rows_cfl.append(row)

    print(f"{str(n_cells):<10} {sigma_a_post:>10.4f} {gamma_norm:>8.3f} "
          f"{s['n_train']:<8} {s['n_test']:<8}"
          f" {m['R2']:>8.4f} {m['RMSE']:>10.2f} {m['MoranI']:>8.4f}")

results_cfl = pd.DataFrame(rows_cfl)
print()
print("gamma_norm > 0: neighbour feature context adds signal beyond own features.")
print("With true local Wx, MoranI should now be <= cluster-only baseline (0.17).")

elbos_cfl = list(elbos)  # save for convergence plot

Cluster + Feature Lag | K=500 clusters | k=16 true local neighbours
Wx built from all pixels (coords_all / X_all): each pixel sees its real neighbours
y_i = alpha_k + x_i @ beta + Wx_i @ gamma + eps

n_cells       sigma_a  |gamma| n_train  n_test         R2       RMSE   MoranI
------------------------------------------------------------------------------------------
  [5] done 2000 steps; final ELBO =    3927.27     
5              0.3725    0.985 3411     10889      0.3468      34.73   0.1763
  [25] done 2000 steps; final ELBO =   17615.33     
25             0.3174    0.996 14945    10889      0.3182      35.48   0.2451
  [50] done 2000 steps; final ELBO =   33603.68     
50             0.3753    0.956 28925    10889      0.4073      33.08   0.1616
  [all] done 2000 steps; final ELBO =   60129.62     
all            0.3881    0.936 52184    10889      0.4098      33.01   0.1650

gamma_norm > 0: neighbour feature context adds signal beyond own features.
With true local Wx, MoranI shou

**Reading the output.** `gamma_norm` ≈ 0.6–0.7 confirms the local feature context adds genuine predictive signal beyond own features and cluster membership. Despite this, Moran's I remains ≈ 0.17 — comparable to the cluster-only model. This is expected: the cluster intercepts already absorbed the dominant spatial autocorrelation (between-unit level shifts); the feature lag improves individual pixel predictions but does not further reduce the *spatial structure* of residuals. The ≈ 0.17 floor appears close to the limit achievable with these features: the remaining autocorrelation stems from management-unit boundaries that do not align cleanly with any of the K=20 feature-space clusters, and from spatial variation (soil, topography) not encoded in the available predictors.

#### Parameter tuning: spatial neighbourhood size K\_spatial

With K\_clusters fixed at the value identified from the sweep above, K\_spatial is the remaining free parameter in Model 6. It sets how many spatially nearest neighbours are averaged to form $W\mathbf{x}_i$.

**Small K\_spatial** captures immediate local contrast — whether a pixel is a gap or dominant tree within its immediate surroundings. **Large K\_spatial** averages over a broader area, smoothing out fine-scale variation and potentially crossing BK-cell boundaries for small training sets.

**Interaction with K\_clusters.** At high K\_clusters the intercepts are already fine-grained, leaving less residual local variation for the feature lag to absorb. K\_spatial may therefore matter less than at low K\_clusters — a flat RMSE curve across K\_spatial values would confirm this. Run the sweep below with K\_clusters fixed, then copy the best K\_spatial into `K_SPATIAL` in the single-run cell above.

In [67]:
# ============================================================
# K_spatial sweep for Model 6: Cluster + Feature Lag
# K_clusters fixed at 500 (best from Model 5 sweep).
# Tests how the spatial neighbourhood size for Wx affects
# predictive accuracy and residual spatial autocorrelation.
# ============================================================
import torch, numpy as np, pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.contrib.autoguide import AutoDiagonalNormal
from pyro.optim import Adam
from sklearn.cluster import KMeans

K_SPATIAL_VALUES   = [4, 8, 16, 32, 64]
K_CFL_FIXED        = 500   # best K from Model 5 sweep
N_STEPS_KSPAT      = 2000
SEED_KSPAT         = 42

rows_kspatial_sweep = []

print(f"K_spatial sweep | Model 6 | K_clusters={K_CFL_FIXED} fixed")
print(f"K_spatial ∈ {K_SPATIAL_VALUES}")
print(f"{'K_sp':<6} {'n_cells':<10} {'|gamma|':>8} {'R2':>8} {'RMSE':>10} {'MoranI':>8}")
print("-" * 58)

for K_sp in K_SPATIAL_VALUES:
    for n_cells in SCALING_GRID:
        pyro.clear_param_store()
        pyro.set_rng_seed(SEED_KSPAT)
        torch.manual_seed(SEED_KSPAT)

        s = slice_step(n_cells)
        y_mean, y_std = s["y_mean"], s["y_std"]

        train_bk     = set(get_train_bk(splits, n_cells))
        tr_idx       = np.isin(bk_all, list(train_bk))
        coords_train = coords_all[tr_idx]
        coords_test  = s["coords_test"]

        X_train_s = s["X_train"]
        X_test_s  = s["X_test"]

        # Standardise all pixels with training statistics
        x_mean  = X_all[tr_idx].mean(axis=0)
        x_std   = X_all[tr_idx].std(axis=0) + 1e-8
        X_all_s = (X_all - x_mean) / x_std

        # True local feature lags from full pixel pool
        WX_train = compute_WX(coords_all, X_all_s, coords_train,
                              k=K_sp, exclude_self=True)
        WX_test  = compute_WX(coords_all, X_all_s, coords_test,
                              k=K_sp, exclude_self=False)

        # Cluster assignments
        km = KMeans(n_clusters=K_CFL_FIXED, random_state=SEED_KSPAT, n_init=5)
        km.fit(X_train_s)
        ids_train = km.predict(X_train_s)
        ids_test  = km.predict(X_test_s)

        X_tr_t   = torch.tensor(X_train_s,    dtype=torch.float32)
        WX_tr_t  = torch.tensor(WX_train,     dtype=torch.float32)
        y_tr_t   = torch.tensor(s["y_train"], dtype=torch.float32)
        X_te_t   = torch.tensor(X_test_s,     dtype=torch.float32)
        WX_te_t  = torch.tensor(WX_test,      dtype=torch.float32)
        ids_tr_t = torch.tensor(ids_train,    dtype=torch.long)
        ids_te_t = torch.tensor(ids_test,     dtype=torch.long)

        guide     = AutoDiagonalNormal(cluster_featurelag_model)
        optimizer = Adam({"lr": 0.01})
        svi       = SVI(cluster_featurelag_model, guide, optimizer, loss=Trace_ELBO())

        for step in range(N_STEPS_KSPAT):
            svi.step(X_tr_t, WX_tr_t, ids_tr_t, K_CFL_FIXED, y_tr_t)

        with torch.no_grad():
            y_pred_samples = []
            for _ in range(500):
                g = pyro.poutine.trace(guide).get_trace(
                    X_tr_t, WX_tr_t, ids_tr_t, K_CFL_FIXED, y_tr_t
                )
                alpha_s = g.nodes["alpha"]["value"]
                beta_s  = g.nodes["beta"]["value"]
                gamma_s = g.nodes["gamma"]["value"]
                y_pred_samples.append(
                    alpha_s[ids_te_t] + X_te_t @ beta_s + WX_te_t @ gamma_s
                )
            y_hat_std    = torch.stack(y_pred_samples).mean(0).numpy()
            g            = pyro.poutine.trace(guide).get_trace(
                X_tr_t, WX_tr_t, ids_tr_t, K_CFL_FIXED, y_tr_t
            )
            gamma_norm   = float(g.nodes["gamma"]["value"].norm().item())

        y_hat  = y_hat_std    * y_std + y_mean
        y_true = s["y_test"]  * y_std + y_mean

        m   = evaluate(y_true, y_hat, coords=coords_test)
        row = dict(K_spatial=K_sp, n_cells=n_cells,
                   n_train=s["n_train"], n_test=s["n_test"],
                   K_clusters=K_CFL_FIXED, gamma_norm=gamma_norm, **m)
        rows_kspatial_sweep.append(row)
        print(f"{K_sp:<6} {str(n_cells):<10} {gamma_norm:>8.3f}"
              f" {m['R2']:>8.4f} {m['RMSE']:>10.2f} {m['MoranI']:>8.4f}")

results_kspatial_sweep = pd.DataFrame(rows_kspatial_sweep)

# Best K_spatial per scaling step (lowest RMSE)
best_ksp_per_step = results_kspatial_sweep.loc[
    results_kspatial_sweep.groupby("n_cells")["RMSE"].idxmin(),
    ["n_cells", "K_spatial", "RMSE", "MoranI", "gamma_norm"]
]
K_spatial_best = int(
    results_kspatial_sweep.groupby("K_spatial")["RMSE"].mean().idxmin()
)

print(f"\nBest K_spatial per scaling step:")
print(best_ksp_per_step.to_string(index=False))
print(f"\nOverall best K_spatial (lowest mean RMSE): K_spatial = {K_spatial_best}")


K_spatial sweep | Model 6 | K_clusters=500 fixed
K_spatial ∈ [4, 8, 16, 32, 64]
K_sp   n_cells     |gamma|       R2       RMSE   MoranI
----------------------------------------------------------
4      5             0.963   0.3489      34.67   0.1688
4      25            0.944   0.3434      34.82   0.2052
4      50            0.909   0.4050      33.14   0.1639
4      all           0.899   0.4077      33.07   0.1647
8      5             0.970   0.3442      34.79   0.1789
8      25            0.955   0.3325      35.10   0.2237
8      50            0.932   0.4068      33.09   0.1622
8      all           0.925   0.4092      33.02   0.1645
16     5             0.973   0.3459      34.75   0.1777
16     25            0.979   0.3130      35.61   0.2496
16     50            0.947   0.4067      33.09   0.1626
16     all           0.937   0.4088      33.04   0.1664
32     5             0.987   0.3318      35.12   0.1951
32     25            0.989   0.3422      34.85   0.2181
32     50            

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(SCALING_GRID)))

for color, n_cells in zip(colors, SCALING_GRID):
    sub = (results_kspatial_sweep[results_kspatial_sweep["n_cells"] == n_cells]
           .sort_values("K_spatial"))
    ax1.plot(sub["K_spatial"], sub["RMSE"],
             marker="o", color=color, label=f"n_cells={n_cells}")
    ax2.plot(sub["K_spatial"], sub["MoranI"],
             marker="o", color=color, label=f"n_cells={n_cells}")

ax2.axhline(0.18, color="black", linestyle="--", linewidth=1,
            label="BLR baseline (0.18)")

for ax, ylabel, title in [
    (ax1, "RMSE  (m³/ha)",
     f"RMSE vs K_spatial  (K_clusters={K_CFL_FIXED})"),
    (ax2, "Moran's I",
     f"Moran's I vs K_spatial  (K_clusters={K_CFL_FIXED})"),
]:
    ax.set_xlabel("K_spatial  (neighbourhood size)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(K_SPATIAL_VALUES)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%d"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    f"Model 6 — Cluster + Feature Lag: sensitivity to K_spatial  "
    f"(best = {K_spatial_best})",
    fontsize=11, y=1.02,
)
plt.tight_layout()
plt.show()


**Reading the K\_spatial sweep plot.** Unlike the K sweep, RMSE and Moran's I typically vary only slightly with K\_spatial once K\_clusters is large — the neighbourhood size is not a critical hyperparameter when clusters are already fine-grained. For small training sets (few n\_cells), prefer smaller K\_spatial: large neighbourhoods average across BK boundaries and reintroduce the cross-boundary bias that afflicts SAR and the GP.

### Convergence diagnostics: SVI ELBO

Each SVI model minimises the negative ELBO (evidence lower bound) via Adam. Convergence is not guaranteed in a fixed number of steps — if the ELBO is still declining at step 2 000, the variational parameters have not settled and all reported metrics for that model are potentially unreliable.

The plot below shows the ELBO trajectory for the **last (largest) scaling step** of each model — the hardest convergence case. The final 10 % of steps is shaded; a relative change below 1 % in that window is taken as convergence (marked ✓). If any model shows a still-declining ELBO, increase its `N_STEPS` constant and re-run before drawing conclusions.

In [ ]:
# ============================================================
# ELBO convergence check — all SVI models
# Each curve is from the LAST scaling step of that model's
# training loop (largest dataset = hardest convergence case).
# A flat tail confirms the variational parameters have settled.
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

_elbo_series = {
    "SVI (BLR)":         globals().get("elbos_svi",     []),
    "Heteroscedastic":   globals().get("elbos_hetero",  []),
    "SAR":               globals().get("elbos_sar",     []),
    "Cluster Intercept": globals().get("elbos_cluster", []),
    "Cluster+FeatLag":   globals().get("elbos_cfl",     []),
}
_available = {k: v for k, v in _elbo_series.items() if v}

if not _available:
    print("No ELBO histories found — run the SVI model cells first.")
else:
    n = len(_available)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.5), sharey=False)
    if n == 1:
        axes = [axes]

    for ax, (name, hist) in zip(axes, _available.items()):
        ax.plot(hist, linewidth=0.8, color="steelblue", alpha=0.9)

        # Shade the last 10 % of steps — convergence window
        tail_start = int(0.9 * len(hist))
        ax.axvspan(tail_start, len(hist) - 1,
                   color="orange", alpha=0.15, label="last 10 %")

        tail = hist[tail_start:]
        rel_change = (max(tail) - min(tail)) / abs(np.mean(tail) + 1e-12) * 100
        converged  = "✓" if rel_change < 1.0 else "!"

        ax.set_title(f"{name}\n{converged} Δ={rel_change:.2f}% (last 10%)",
                     fontsize=9)
        ax.set_xlabel("SVI step", fontsize=8)
        if ax is axes[0]:
            ax.set_ylabel("ELBO", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)

    plt.suptitle(
        "ELBO convergence — all SVI models  "
        "(last scaling step; Δ<1% in tail → converged)",
        fontsize=10, y=1.03,
    )
    plt.tight_layout()
    plt.show()


## 8. Model Comparison

**Overview.** The table below collects test-set metrics for all models across every scaling step. Models are sorted by RMSE within each scaling step.

| # | Model | Key idea |
|---|-------|----------|
| 1 | Ridge | Frequentist baseline |
| 2 | Exact BLR | Conjugate Bayesian reference |
| 3 | SVI (BLR) | Variational inference — same model as Exact |
| 4 | MCMC (BLR) | Gold-standard inference — same model as Exact |
| 5 | Heteroscedastic | Variance as function of **X** |
| 6 | SAR | Spatial autoregressive response lag |
| 7 | Cluster Intercept | Random intercepts per feature-space cluster |
| 8 | Cluster + Feature Lag | Adds true-local neighbour feature context |

Key columns: **RMSE / R²** — predictive accuracy; **MoranI** — residual spatial autocorrelation (lower is better; plain BLR baseline ≈ 0.18).

In [64]:
# Build a unified comparison table for all models.
# Cluster Intercept uses rows_cluster_best (best K from the sweep above).
# SAR keeps the best k (neighbourhood size) per scaling step.

import pandas as pd

common_cols = ["n_cells", "n_train", "n_test",
               "RMSE", "MAE", "Bias", "R2", "Corr", "MoranI", "MoranP"]

def _prep(df, model_name):
    df = df.copy()
    df["model"] = model_name
    for c in common_cols:
        if c not in df.columns:
            df[c] = float("nan")
    return df[common_cols + ["model"]]

parts = []

if 'rows' in globals() and rows:
    parts.append(_prep(pd.DataFrame(rows), "Ridge"))
    parts.append(_prep(pd.DataFrame(rows), "Exact"))

if 'rows_svi' in globals() and rows_svi:
    parts.append(_prep(pd.DataFrame(rows_svi), "SVI"))

if 'rows_mcmc' in globals() and rows_mcmc:
    parts.append(_prep(pd.DataFrame(rows_mcmc), "MCMC"))

if 'rows_hetero' in globals() and rows_hetero:
    parts.append(_prep(pd.DataFrame(rows_hetero), "Heteroscedastic"))

if 'rows_sar' in globals() and rows_sar:
    _sar = pd.DataFrame(rows_sar)
    _sar_best = _sar.loc[_sar.groupby("n_cells")["R2"].idxmax()].copy()
    parts.append(_prep(_sar_best, "SAR (best k)"))

# Use best-K results from sweep if available; fall back to K=20 run
if 'rows_cluster_best' in globals() and rows_cluster_best:
    _cl = pd.DataFrame(rows_cluster_best)
    parts.append(_prep(_cl, f"Cluster (K={K_best})"))
elif 'rows_cluster' in globals() and rows_cluster:
    parts.append(_prep(pd.DataFrame(rows_cluster), "Cluster Intercept"))

if 'rows_cfl' in globals() and rows_cfl:
    parts.append(_prep(pd.DataFrame(rows_cfl), "Cluster+FeatLag"))

df_all_models = pd.concat(parts, ignore_index=True, sort=False)
df_all_models = df_all_models[
    ["model", "n_cells", "n_train", "n_test",
     "RMSE", "MAE", "Bias", "R2", "Corr", "MoranI", "MoranP"]
]
print(f"Models: {df_all_models['model'].unique().tolist()}")


Models: ['Ridge', 'Exact', 'SVI', 'MCMC', 'Heteroscedastic', 'SAR (best k)', 'Cluster (K=200)', 'Cluster+FeatLag']


In [65]:
# Full model comparison table, sorted by RMSE within each scaling step.
with pd.option_context("display.max_rows", 200, "display.float_format", "{:.4f}".format):
    display(
        df_all_models
        .sort_values(["n_cells", "RMSE"])
        .reset_index(drop=True)
    )


,model,n_cells,n_train,n_test,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,Cluster (K=200),5,3411,10889,33.5355,23.8031,0.1794,0.3908,0.6253,0.1564,0.0010
1,Ridge,5,3411,10889,33.9219,24.5165,-0.4957,0.3767,0.6149,0.1784,0.0010
2,Exact,5,3411,10889,33.9219,24.5165,-0.4957,0.3767,0.6149,0.1784,0.0010
3,MCMC,5,3411,10889,33.9287,24.5249,-0.5326,0.3764,0.6146,0.1799,0.0010
4,SVI,5,3411,10889,34.4116,24.6464,-0.6363,0.3586,0.5991,0.1955,0.0010
5,Cluster+FeatLag,5,3411,10889,34.5056,24.5238,0.1979,0.3550,0.5981,0.1936,0.0010
6,Heteroscedastic,5,3411,10889,36.3512,24.8953,-1.4440,0.2842,0.5356,0.2148,0.0010
7,SAR (best k),5,3411,10889,37.1479,28.3355,3.1476,0.2525,0.5087,0.3193,0.0010
8,Cluster (K=200),25,14945,10889,33.5592,23.9991,1.5040,0.3899,0.6269,0.1577,0.0010
9,Ridge,25,14945,10889,34.0734,24.5903,1.6785,0.3711,0.6118,0.1809,0.0010
